# Carbon stock modelling and mapping in Malagasy open ecosystems — Python / Kaggle edition (v4)
**RF vs GAM vs LMM · site-specific spatial blocks · nested spatial block CV (10 outer / 5 inner) · Optuna tuning · 100 bootstrap maps**

This notebook is the Python equivalent of `03_nested_spatial_RF_GAM_LMM_manuscript_v4.qmd` and is built for a Kaggle CPU notebook.

| Step | R version | This notebook |
|---|---|---|
| Scales / pools | ALL + AF, AT, IV; 7 pools | same |
| Covariates | 13 rasters per site, extracted at plots | same (`rasterio`); NA gaps filled from the nearest valid cell |
| Spatial blocks | `blockCV` (square 11 km AF, 3.5 km AT, 1.5 km IV); ALL = union of site folds | re-implemented (`numpy`/`scipy`); same designs, same search criteria; optional import of the **exact R folds** |
| Nested CV | 10 outer / 5 inner, inner folds rebuilt inside each outer training set | same |
| RF | `ranger` + VSURF + grid | `scikit-learn` RF; **VSURF-style selection on inner spatial folds**; **Optuna** (TPE + pruning) for `mtry`, `min_samples_leaf` |
| GAM | `mgcv` | additive **penalised B-spline (P-spline) model**, forward selection by inner-CV, **Optuna** for `k`, smoothing `lambda` |
| LMM | `lme4` | `statsmodels` MixedLM, forward AIC (ML), random intercept `Site` (ALL) or `block` (sites), fixed effects only for prediction |
| Metrics / tests | R², RMSE, nRMSE, MAE, RPD, t-CI, paired Wilcoxon + Holm | same |
| Importance | OOB permutation %IncMSE + selection frequency + partial dependence | same (OOB permutation implemented on the forest's own OOB samples) |
| Uncertainty | B = 100 within-block bootstrap, per-pixel mean/SD/2.5–97.5 %, totals in Tg C | same; GAM/LMM bootstraps are a single matrix product per map chunk; RF bootstraps use small forests |

**How to run on Kaggle**
1. *Add data*: the dataset folder with `Datamapf_coordo.csv` and the folder with the covariate rasters (`<Covariate>_<SITE>.tif`, e.g. `MNT_AF.tif`; sub-folders are fine).
2. *Session options*: CPU (4 cores), Internet on only if a package is missing. Edit the paths in section 1 if your dataset slugs differ.
3. *Optional but recommended for the manuscript*: add `folds_ALL_sites_combined.csv` (produced by the R workflow) as a dataset and set `R_FOLDS_CSV` to its path — the notebook then uses **exactly the same folds as R**.
4. *Run all*. Everything is cached in `/kaggle/working/outputs/cache`; after a timeout simply run again. Smoke test first with `os.environ["MADA_FAST"] = "1"` in the settings cell.

**Efficiency.** Outer folds × pools × scales run in parallel (`joblib`, all cores) and every task is cached on disk (`/kaggle/working/outputs/cache`), so an interrupted session resumes where it stopped. Set `MADA_FAST=1` for a 5-minute smoke test.

**Kaggle settings:** Accelerator *None* (CPU) is enough; *Internet* on only if a package is missing (all imports below are in the default Kaggle image except possibly `optuna`).

In [ ]:
import os, sys, re, math, time, json, hashlib, pickle, warnings, itertools, subprocess
from pathlib import Path
warnings.filterwarnings("ignore"); os.environ["PYTHONWARNINGS"] = "ignore"        # also silences the joblib worker processes

def _need(mod, pip_name=None):
    try: __import__(mod)
    except ImportError:
        print(f"installing {pip_name or mod} ..."); subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or mod], check=False)
for m, p in [("optuna", "optuna"), ("statsmodels", "statsmodels"), ("rasterio", "rasterio"), ("pyproj", "pyproj"), ("shapely", "shapely")]:
    _need(m, p)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from scipy import stats, ndimage
from scipy.spatial import cKDTree
from scipy.spatial.distance import pdist
from scipy.optimize import curve_fit
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.preprocessing import SplineTransformer
import optuna, rasterio, statsmodels.api as sm
from rasterio.warp import reproject, Resampling
from rasterio.transform import Affine
from pyproj import Transformer
from shapely.geometry import MultiPoint
import shapely
from statsmodels.stats.multitest import multipletests
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Settings

In [ ]:
## ============================== USER SETTINGS ====================================
FAST = os.environ.get("MADA_FAST", "0") == "1"                      # quick smoke test
DATA_CSV = Path(os.environ.get("MADA_CSV", "/kaggle/input/datasets/antsasarobidyran/dataset/Datamapf_coordo.csv"))
COV_DIR  = Path(os.environ.get("MADA_COV", "/kaggle/input/datasets/antsasarobidyran/covariates"))
OUT      = Path(os.environ.get("MADA_OUT", "/kaggle/working/outputs"))
R_FOLDS_CSV = os.environ.get("MADA_RFOLDS", "")                       # optional: path of folds_ALL_sites_combined.csv from the R workflow -> exactly the same folds as R
N_JOBS   = int(os.environ.get("MADA_JOBS", os.cpu_count() or 1))
SEED     = 2026

COORDS_EPSG, PROJ_EPSG = 4326, 32738                                  # lon/lat of the CSV; UTM 38S for distances, blocks, maps
SITES  = ["AF", "AT", "IV"]; SCALES = ["ALL"] + SITES
TARGETS = {"SOC30": "SOC 0–30 cm", "SOC100": "SOC 0–100 cm", "AGC": "AGC", "BGC": "BGC",
           "DW": "Deadwood", "Litter": "Litter", "TotC": "Total C"}
PRED_CONT = ["Elevation", "Slope", "Edge", "Roughness", "TPI", "TRI", "Cos_aspect", "Sin_aspect", "MAP", "MAT"]
PRED_CAT  = ["LC", "Fire", "Soil"]
ALL_COV   = PRED_CONT + PRED_CAT
COV_ALIAS = {"MNT": "Elevation", "ESA": "LC", "FF": "Fire", "Cosaspect": "Cos_aspect", "Sinaspect": "Sin_aspect"}

FILL_GAPS, NA_RESCUE_M = True, 45                                      # gaps inside the footprint -> nearest valid cell; plot on NA cell -> valid cells within 45 m
MAP_AGG_FACT = 3                                                       # 1 = native 30 m; 3 = 90 m (about 9x faster)

# ---- spatial blocks (same designs as the R v4 workflow)
K_OUTER, K_INNER, BLOCK_ITER = 10, 5, 100
BLOCK_MODE = os.environ.get("MADA_BLOCK_MODE", "fixed")                                                 # "fixed" = designs found by the R search | "search" = re-run the design search here
BLOCK_FIXED = {"AF": dict(type="spatial", hexagon=False, size=11000, seed=1),
               "AT": dict(type="spatial", hexagon=False, size=3500,  seed=3),
               "IV": dict(type="spatial", hexagon=False, size=1500,  seed=2)}
AUTOCORR_RANGE_M = {"AF": 1930, "AT": 340, "IV": 260}                   # median autocorrelation range of the 7 pools (R / automap estimate); None -> estimated here
INNER_SEED, SEARCH_NSEED, SEARCH_TOL, SEARCH_MIN_NMIN, HULL_BUFFER, MAX_PRED_CELLS = 1, 5, 0.03, 4, 0.10, 2500

# ---- models
NUM_TREES, NUM_TREES_INNER, NUM_TREES_BOOT = 500, 150, 50              # final RF / inner tuning / each bootstrap RF (B x 50 trees)
MTRY_RANGE, MSL_CHOICES = (2, 13), [1, 3, 5, 10]
N_TRIALS_RF, N_TRIALS_GAM, K_GAM = 20, 20, [3, 5, 8]
SEL_MIN_GAIN, LMM_DELTA_AIC = 0.005, 2.0                               # forward-selection stopping rules (GAM: relative RMSE gain, LMM: AIC drop)
SEL_FREQ_MIN, IMP_REL_MIN = 0.5, 0.10                                  # main predictor: selected in >= 50 % of outer sets and %IncMSE >= 10 % of the max
B_BOOT, CLIP_NEGATIVE, CHUNK_CELLS = 100, True, 100_000
FIG_DPI = 600

if FAST:
    K_OUTER, K_INNER, B_BOOT = 4, 3, 6
    NUM_TREES, NUM_TREES_INNER, NUM_TREES_BOOT, N_TRIALS_RF, N_TRIALS_GAM = 80, 40, 20, 4, 4
    TARGETS = {k: TARGETS[k] for k in ["SOC30", "TotC"]}
## ==================================================================================
ALGOS = ["RF", "GAM", "LMM"]
OUT.mkdir(parents=True, exist_ok=True)
for sub in ["cache", "figures", "maps", "tables"]: (OUT / sub).mkdir(exist_ok=True)
CACHE, FIG, MAPS, TAB = OUT / "cache", OUT / "figures", OUT / "maps", OUT / "tables"
CODE_VERSION = "py-v4.1"                                               # bump to invalidate every cached result after editing the code
CFG_TAG = hashlib.md5(json.dumps([CODE_VERSION, NUM_TREES, NUM_TREES_INNER, NUM_TREES_BOOT, MTRY_RANGE, MSL_CHOICES, N_TRIALS_RF, N_TRIALS_GAM, K_GAM,
                                  SEL_MIN_GAIN, LMM_DELTA_AIC, K_OUTER, K_INNER, B_BOOT, CLIP_NEGATIVE, list(TARGETS)]).encode()).hexdigest()[:8]
print(f"FAST={FAST} | cores={N_JOBS} | outer/inner = {K_OUTER}/{K_INNER} | out = {OUT}")

def cached(name, fn):
    f = CACHE / f"{name}.pkl"
    if f.exists():
        with open(f, "rb") as fh: return pickle.load(fh)
    val = fn()
    with open(f, "wb") as fh: pickle.dump(val, fh)
    return val

# figure style: Arial if present (Liberation Sans is metric-identical), else DejaVu
from matplotlib import font_manager as _fm
_names = {f.name for f in _fm.fontManager.ttflist}
FONT = next((f for f in ["Arial", "Liberation Sans", "Helvetica", "DejaVu Sans"] if f in _names), "DejaVu Sans")
plt.rcParams.update({"font.family": FONT, "font.size": 10, "axes.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
                     "figure.dpi": 110, "savefig.dpi": FIG_DPI, "savefig.bbox": "tight", "axes.titlesize": 11, "axes.labelsize": 10})
PAL_SCALE = {"ALL": "#3C5488", "AF": "#E64B35", "AT": "#4DBBD5", "IV": "#00A087"}
PAL_MOD   = {"RF": "#3C5488", "GAM": "#F39B7F", "LMM": "#00A087"}
FOLD_COLS = ["#4E79A7", "#F28E2B", "#E15759", "#76B7B2", "#59A14F", "#EDC948", "#B07AA1", "#FF9DA7", "#9C755F", "#8C8C8C"]
def save_fig(fig, name, dpi=None):
    fig.savefig(FIG / f"{name}.png", dpi=dpi or FIG_DPI); fig.savefig(FIG / f"{name}.pdf")

# 2. Data
`Datamapf_coordo.csv` provides the pools, the site and the coordinates. **Covariates are not taken from the CSV**: they are extracted from the rasters (next section), exactly as in the R workflow.

In [ ]:
def read_table(path):
    head = open(path, encoding="utf-8-sig").read().splitlines()[:6]
    delim = max([";", ",", "\t"], key=lambda c: head[0].count(c))
    dec = "," if delim == ";" and any(re.search(r"[0-9],[0-9]", l) for l in head[1:]) else "."
    return pd.read_csv(path, sep=delim, decimal=dec, skipinitialspace=True, encoding="utf-8-sig")

norm = lambda x: re.sub(r"[^A-Za-z0-9]", "", str(x)).upper()
raw = read_table(DATA_CSV)
aliases = {"SOC30": ["ST30"], "SOC100": ["ST100"], "AGC": ["AGCGIEC"], "BGC": ["BGCGIEC"], "DW": ["DWGIEC"],
           "Litter": ["LITERGIEC", "LITTERGIEC"], "TotC": ["TOTC"], "Name": [], "Site": [], "X": [], "Y": []}
ren = {}
for canon, alts in aliases.items():
    hit = [c for c in raw.columns if norm(c) in {norm(canon)} | {norm(a) for a in alts}]
    if hit: ren[hit[0]] = canon
raw = raw.rename(columns=ren)
miss = [c for c in ["Name", "Site", "X", "Y", *TARGETS] if c not in raw.columns]
assert not miss, f"Columns missing in the CSV: {miss}\nFound: {list(raw.columns)}"
cov_norm = {norm(c) for c in ALL_COV + list(COV_ALIAS) + ["Aspect", "NDVI"]}
csv_cov = [c for c in raw.columns if norm(c) in cov_norm]
if csv_cov: print("CSV covariate columns ignored (values come from the rasters):", csv_cov)
dat = raw.drop(columns=csv_cov).copy()
dat["Site"] = dat["Site"].astype(str).str.strip().str.upper()
assert set(dat["Site"]) <= set(SITES), f"Unknown Site values: {set(dat['Site']) - set(SITES)}"
dat["plot_id"] = dat["Name"].astype(str).str.strip()
assert not dat["plot_id"].duplicated().any() and dat[["X", "Y"]].notna().all().all()
_t = Transformer.from_crs(COORDS_EPSG, PROJ_EPSG, always_xy=True)
dat["px"], dat["py"] = _t.transform(dat["X"].values, dat["Y"].values)
dat = dat.reset_index(drop=True)
display(dat.groupby("Site").size().rename("n_plots").to_frame().T)
print("max |TotC - (SOC100+AGC+BGC+DW+Litter)| =", np.abs(dat.TotC - (dat.SOC100 + dat.AGC + dat.BGC + dat.DW + dat.Litter)).max().round(3))
def extent_diag(d): return float(np.hypot(np.ptp(d["px"]), np.ptp(d["py"])))

# 3. Covariate rasters and extraction at the plots
One set of rasters per site on a shared grid. Gaps inside the study-area footprint (`Sin_aspect` valid cells) are filled from the **nearest valid cell** (one `distance_transform_edt` pass instead of the iterative focal filter of the R version). A plot that falls on a `NA` cell takes the mean (continuous) / most frequent class (categorical) of the valid cells within `NA_RESCUE_M` metres.

In [ ]:
RAST_ALIAS = {k.upper(): v for k, v in COV_ALIAS.items()} | {c.upper(): c for c in ALL_COV}
all_tifs = sorted(p for p in COV_DIR.rglob("*") if p.suffix.lower() in (".tif", ".tiff"))
print(f"{len(all_tifs)} rasters under {COV_DIR}")

def raster_inventory(site):
    """<Covariate>_<SITE>.tif (also <SITE>_<Covariate>, or <SITE>/<Covariate>.tif); a trailing resolution tag such as _30m is ignored"""
    inv = {}
    for p in all_tifs:
        m = re.match(rf"^(.*)[_-]{site}$", p.stem, flags=re.I) or re.match(rf"^{site}[_-](.*)$", p.stem, flags=re.I)
        key = m.group(1) if m else (p.stem if p.parent.name.upper() == site else None)
        if key is None: continue
        cov = RAST_ALIAS.get(re.sub(r"[_-]\d+m$", "", key, flags=re.I).upper())
        if cov is None: continue
        assert cov not in inv, f"duplicated raster for {site}/{cov}: {p}"
        inv[cov] = p
    assert inv, f"no raster named <covariate>_{site}.tif under {COV_DIR}"
    return inv

def read_layer(path, ref=None, categorical=False):
    with rasterio.open(path) as src:
        a = src.read(1, masked=True).astype("float32").filled(np.nan)
        if ref is not None and (src.transform != ref["transform"] or a.shape != ref["shape"]):
            out = np.full(ref["shape"], np.nan, "float32")
            reproject(a, out, src_transform=src.transform, src_crs=src.crs, dst_transform=ref["transform"], dst_crs=ref["crs"],
                      resampling=Resampling.nearest if categorical else Resampling.bilinear, src_nodata=np.nan, dst_nodata=np.nan)
            a = out
        meta = dict(transform=src.transform, crs=src.crs, shape=a.shape)
    return a, meta

def fill_from_nearest(a, footprint):
    gap = footprint & np.isnan(a)
    if not gap.any(): return a, 0, 0
    n0 = int(gap.sum())
    idx = ndimage.distance_transform_edt(np.isnan(a), return_distances=False, return_indices=True)
    filled = a[idx[0], idx[1]]
    a = np.where(gap, filled, a)
    a[~footprint] = np.nan
    return a.astype("float32"), n0, int((footprint & np.isnan(a)).sum())

def load_site_rasters(site):
    inv = raster_inventory(site)
    ref_name = "Sin_aspect" if "Sin_aspect" in inv else next(iter(inv))
    a0, meta = read_layer(inv[ref_name])
    layers = {ref_name: a0}
    for cov, p in inv.items():
        if cov != ref_name: layers[cov], _ = read_layer(p, ref=meta, categorical=cov in PRED_CAT)
    foot = ~np.isnan(layers[ref_name]); rep = []
    for cov in list(layers):
        layers[cov], n0, n1 = fill_from_nearest(layers[cov], foot) if FILL_GAPS else (layers[cov], 0, 0)
        rep.append(dict(Site=site, Covariate=cov, Footprint_cells=int(foot.sum()), NA_inside_footprint=n0, NA_after_fill=n1))
    return dict(layers=layers, footprint=foot, **meta), rep

RAST, fill_rows = {}, []
for s in SITES:
    RAST[s], r = load_site_rasters(s); fill_rows += r
    lay = RAST[s]["layers"]; res = RAST[s]["transform"].a
    print(f"{s}: {len(lay)} layers {RAST[s]['shape']} res={res:.6f} deg; missing: {sorted(set(ALL_COV) - set(lay)) or 'none'}")
fill_report = pd.DataFrame(fill_rows)
display(fill_report[fill_report.NA_inside_footprint > 0])
COV_AVAIL = {s: [c for c in ALL_COV if c in RAST[s]["layers"]] for s in SITES}
def scale_cov(sc): return [c for c in ALL_COV if all(c in COV_AVAIL[s] for s in (SITES if sc == "ALL" else [sc]))]

def extract_at_plots(site):
    d = dat[dat.Site == site]; R = RAST[site]; t = R["transform"]; H, W = R["shape"]
    rows, cols = rasterio.transform.rowcol(t, d["X"].values, d["Y"].values)
    rows, cols = np.asarray(rows), np.asarray(cols); inside = (rows >= 0) & (rows < H) & (cols >= 0) & (cols < W)
    ex = {}
    for cov, a in R["layers"].items():
        v = np.full(len(d), np.nan); v[inside] = a[rows[inside], cols[inside]]; ex[cov] = v
    ex = pd.DataFrame(ex, index=d.index); before = ex.isna().sum()
    bad = np.flatnonzero(ex.isna().any(axis=1).values)
    if len(bad) and NA_RESCUE_M > 0:
        lat = d["Y"].values; dx = abs(t.a) * 111320 * np.cos(np.radians(lat.mean())); dy = abs(t.e) * 110574
        rr, rc = int(np.ceil(NA_RESCUE_M / dy)), int(np.ceil(NA_RESCUE_M / dx))
        for k in bad:
            r0, c0 = rows[k], cols[k]
            r1, r2, c1, c2 = max(r0 - rr, 0), min(r0 + rr + 1, H), max(c0 - rc, 0), min(c0 + rc + 1, W)
            yy, xx = np.mgrid[r1:r2, c1:c2]; m = np.hypot((yy - r0) * dy, (xx - c0) * dx) <= NA_RESCUE_M
            for cov in ex.columns[ex.iloc[k].isna()]:
                vals = R["layers"][cov][r1:r2, c1:c2][m]; vals = vals[~np.isnan(vals)]
                if len(vals): ex.iloc[k, ex.columns.get_loc(cov)] = (pd.Series(vals).mode().iloc[0] if cov in PRED_CAT else vals.mean())
    return ex, before, ex.isna().sum()

ex_parts, na_rows = [], []
for s in SITES:
    ex, b, a = extract_at_plots(s); ex_parts.append(ex)
    na_rows += [dict(Site=s, Covariate=c, NA_before_rescue=int(b[c]), NA_after_rescue=int(a[c])) for c in ex.columns if b[c] or a[c]]
cov_df = pd.concat(ex_parts).reindex(dat.index)
for c in ALL_COV:
    if c not in cov_df: cov_df[c] = np.nan
dat = pd.concat([dat.drop(columns=[c for c in ALL_COV if c in dat]), cov_df[ALL_COV]], axis=1)
for v in PRED_CAT: dat[v] = np.where(dat[v].notna(), dat[v].round(), np.nan)          # raster class codes
if na_rows: display(pd.DataFrame(na_rows))

def scale_data(sc):
    d = dat if sc == "ALL" else dat[dat.Site == sc]
    return d.dropna(subset=scale_cov(sc)).copy()

qa = pd.DataFrame([dict(Site=s, Covariate=c, type="categorical" if c in PRED_CAT else "continuous", n=int((dat.Site == s).sum()),
                        n_NA=int(dat.loc[dat.Site == s, c].isna().sum()), min=dat.loc[dat.Site == s, c].min(),
                        median=dat.loc[dat.Site == s, c].median(), max=dat.loc[dat.Site == s, c].max(),
                        n_classes=dat.loc[dat.Site == s, c].nunique() if c in PRED_CAT else np.nan)
                   for s in SITES for c in COV_AVAIL[s]])
display(qa.round(3))

# 4. Predictors and collinearity
All covariates with a raster at the modelling scale are kept; collinearity is **documented, not pruned** (VIF, Spearman). Only constants are dropped.

In [ ]:
PRED_SETS = {}
for sc in SCALES:
    d = scale_data(sc); av = scale_cov(sc)
    cont = [v for v in PRED_CONT if v in av and d[v].std() > 0]
    cat = [v for v in PRED_CAT if v in av and d[v].nunique() > 1]
    PRED_SETS[sc] = dict(cont=cont, cat=cat)
display(pd.DataFrame([dict(Scale=sc, n=len(scale_data(sc)), Continuous=", ".join(PRED_SETS[sc]["cont"]), Categorical=", ".join(PRED_SETS[sc]["cat"])) for sc in SCALES]))

vif_rows, rho_rows = [], []
for sc in SCALES:
    X = scale_data(sc)[PRED_SETS[sc]["cont"]]
    try: v = np.diag(np.linalg.inv(np.corrcoef(X.values.T)))
    except np.linalg.LinAlgError: v = np.full(X.shape[1], np.nan)
    vif_rows += [dict(Scale=sc, Covariate=c, VIF=x) for c, x in zip(X.columns, v)]
    R_ = X.corr(method="spearman").where(np.tril(np.ones((X.shape[1],) * 2), -1).astype(bool))
    rho_rows += [dict(Scale=sc, Covariate_1=a, Covariate_2=b, Spearman=R_.loc[a, b]) for a in R_.index for b in R_.columns if abs(R_.loc[a, b]) > 0.8]
vif_wide = pd.DataFrame(vif_rows).pivot(index="Covariate", columns="Scale", values="VIF").reset_index()
rho_tab = pd.DataFrame(rho_rows, columns=["Scale", "Covariate_1", "Covariate_2", "Spearman"])
display(vif_wide.round(1)); display(rho_tab.round(2))

# 5. Spatial block design
**One design per site**, identical for the 7 pools and the 3 algorithms; a design is judged on **geometry only** (never on model scores):

| Criterion | Meaning | Wanted |
|---|---|---|
| `W_rel` | Wasserstein distance between the CV test→train nearest distances and the map-cell→nearest-plot distances (NNDM logic), divided by the median of the latter | small |
| `leak` | share of test plots with a training plot closer than the autocorrelation range | small |
| `n_min`, `n_cv` | smallest outer test fold; CV of fold sizes | n_min ≥ 4, small |
| `env_infl` | covariate-space extrapolation vs a random k-fold | not ≫ 1 |

`BLOCK_MODE = "fixed"` uses the designs found by the R search (square 11 km AF, 3.5 km AT, 1.5 km IV); `"search"` re-runs the search here. **Nested folds:** outer = design with K = 10; inner folds are **rebuilt inside each outer training set** (blocks shrunk by 15 % if needed); **ALL = union of the site folds**. Python and R random number generators differ, so the fold *assignment* differs from R unless you provide `MADA_RFOLDS` (the `folds_ALL_sites_combined.csv` of the R workflow), which is then used as is.

In [ ]:
# ---------------------------------------------------------------- blocks
def square_block_ids(xy, size):
    o = xy.min(axis=0); c = np.floor((xy - o) / size).astype(int)
    return np.array([f"s{a}_{b}" for a, b in c]), dict(kind="square", origin=o, size=size)

def hex_block_ids(xy, size):                         # pointy-top hexagons, flat-to-flat width = size
    s = size / np.sqrt(3); x = (np.sqrt(3) / 3 * xy[:, 0] - xy[:, 1] / 3) / s; z = (2 / 3 * xy[:, 1]) / s; y = -x - z
    rx, ry, rz = np.round(x), np.round(y), np.round(z); dx, dy, dz = abs(rx - x), abs(ry - y), abs(rz - z)
    fx = (dx > dy) & (dx > dz); fy = ~fx & (dy > dz); fz = ~fx & ~fy
    rx = np.where(fx, -ry - rz, rx); rz = np.where(fz, -rx - ry, rz)
    return np.array([f"h{int(a)}_{int(b)}" for a, b in zip(rx, rz)]), dict(kind="hex", size=size)

def assign_blocks(blk, k, seed, iters=None):
    """random, balanced block -> fold assignment (greedy on fold load, best of `iters` random orders)"""
    iters = iters or BLOCK_ITER
    ub, inv, cnt = np.unique(blk, return_inverse=True, return_counts=True)
    if len(ub) < k: raise ValueError("fewer blocks than folds")
    rng = np.random.default_rng(seed); best = None
    for _ in range(iters):
        load = np.zeros(k); fo = np.empty(len(ub), int)
        for j in rng.permutation(len(ub)):
            c = np.flatnonzero(load == load.min()); f = c[rng.integers(len(c))]; fo[j] = f; load[f] += cnt[j]
        sc = load.max() - load.min()
        if best is None or sc < best[0]: best = (sc, fo)
    return best[1][inv] + 1

def build_folds(xy, design, k, seed):
    if design["type"] == "cluster":
        ids = KMeans(k, n_init=20, random_state=seed).fit_predict(xy) + 1
        return dict(ids=ids, blk=np.array([f"c{i}" for i in ids]), size=np.nan, geom=dict(kind="cluster"))
    sz = design["size"]; fn = hex_block_ids if design.get("hexagon") else square_block_ids
    while sz >= 1000:
        try:
            blk, geom = fn(xy, sz); return dict(ids=assign_blocks(blk, k, seed), blk=blk, size=sz, geom=geom)
        except ValueError: sz *= 0.85
    warnings.warn(f"could not build {k} block folds; using spatial k-means")
    ids = KMeans(k, n_init=20, random_state=seed).fit_predict(xy) + 1
    return dict(ids=ids, blk=np.array([f"c{i}" for i in ids]), size=np.nan, geom=dict(kind="cluster"))

def design_label(des):
    return "k-means (xy)" if des["type"] == "cluster" else f"{'hexagon' if des.get('hexagon') else 'square'} {des['size'] / 1000:g} km"

_RF_OK = bool(R_FOLDS_CSV) and Path(R_FOLDS_CSV).exists() and K_OUTER == 10 and K_INNER == 5
_RF = pd.read_csv(R_FOLDS_CSV) if _RF_OK else None
if R_FOLDS_CSV and not _RF_OK: print("R folds not used (file missing or K != 10/5): folds are built in Python.")

def make_plan_site(site, design):
    d = dat[dat.Site == site]; xy = d[["px", "py"]].values; n = len(d)
    if _RF_OK:                                                           # exact folds of the R workflow
        r = _RF[_RF.Site == site].copy(); r.index = r["Name"].astype(str).str.strip(); r = r.reindex(d.plot_id.values)
        assert r["outer"].notna().all(), f"R folds do not cover all plots of {site}"
        blk, geom = square_block_ids(xy, design["size"]) if design["type"] == "spatial" and not design.get("hexagon") else (np.array([f"c{i}" for i in r["outer"]]), dict(kind="cluster"))
        plan = pd.DataFrame({"plot_id": d.plot_id.values, "outer_": r["outer"].astype(int).values, "blk_": [f"{site}_{b}" for b in blk]})
        for f in range(1, K_OUTER + 1): plan[f"inner_o{f}"] = r[f"inner_o{f}"].fillna(0).astype(int).values
        plan["inner_full"] = build_folds(xy, design, K_INNER, INNER_SEED)["ids"]
        return dict(plan=plan, design=design, size_used=design.get("size", np.nan), geom=geom)
    o = build_folds(xy, design, K_OUTER, design["seed"])
    plan = pd.DataFrame({"plot_id": d.plot_id.values, "outer_": o["ids"], "blk_": [f"{site}_{b}" for b in o["blk"]]})
    for f in range(1, K_OUTER + 1):
        tr = plan.outer_.values != f; v = np.zeros(n, int); v[tr] = build_folds(xy[tr], design, K_INNER, INNER_SEED)["ids"]
        plan[f"inner_o{f}"] = v                                          # 0 = plot belongs to the outer test fold
    plan["inner_full"] = build_folds(xy, design, K_INNER, INNER_SEED)["ids"]
    return dict(plan=plan, design=design, size_used=o["size"], geom=o["geom"])

# ---------------------------------------------------------------- geometry / criteria
def nn_dist(a, b): return cKDTree(b).query(a)[0]

_VG = {"Sph": (lambda h, c0, c1, a: np.where(h < a, c0 + c1 * (1.5 * h / a - 0.5 * (h / a) ** 3), c0 + c1), 1.0),
       "Exp": (lambda h, c0, c1, a: c0 + c1 * (1 - np.exp(-h / a)), 3.0),
       "Gau": (lambda h, c0, c1, a: c0 + c1 * (1 - np.exp(-(h / a) ** 2)), np.sqrt(3))}
def fit_variogram(xy, z, nbins=15):
    """empirical variogram (cutoff = diagonal/3) + Sph/Exp/Gau fits by weighted LS (weights N/h^2, as gstat); lowest weighted SSE kept"""
    cutoff = np.hypot(np.ptp(xy[:, 0]), np.ptp(xy[:, 1])) / 3; dd = pdist(xy); gg = 0.5 * pdist(z[:, None], "sqeuclidean"); m = dd <= cutoff
    b = np.digitize(dd[m], np.linspace(0, cutoff, nbins + 1)) - 1; dm, gm = dd[m], gg[m]
    keep = [i for i in range(nbins) if (b == i).sum() >= 5]
    if len(keep) < 4: return dict(model=None, eff_range=np.nan, cutoff=cutoff, h=np.array([]), gamma=np.array([]))
    h = np.array([dm[b == i].mean() for i in keep]); gam = np.array([gm[b == i].mean() for i in keep]); npr = np.array([(b == i).sum() for i in keep])
    w = npr / h ** 2; best = None
    for name, (fn, mult) in _VG.items():
        try:
            p, _ = curve_fit(fn, h, gam, p0=[gam.min() * 0.5, gam.max() * 0.7, cutoff / 3], sigma=1 / np.sqrt(w),
                             bounds=([0, 0, h.min() * 0.5], [gam.max() * 2, gam.max() * 3, cutoff * 5]), maxfev=5000)
        except Exception: continue
        sse = float(np.sum(w * (gam - fn(h, *p)) ** 2))
        if best is None or sse < best["sse"]: best = dict(model=name, sse=sse, nugget=p[0], psill=p[1], range=p[2], eff_range=p[2] * mult, fn=fn, p=p)
    if best is None: return dict(model=None, eff_range=np.nan, cutoff=cutoff, h=h, gamma=gam)
    best.update(cutoff=cutoff, h=h, gamma=gam); return best

def site_range(site):                                                    # median effective range of the log(1+pool) variograms (like cv_spatial_autocor)
    d = dat[dat.Site == site]; xy = d[["px", "py"]].values; dg = extent_diag(d)
    if AUTOCORR_RANGE_M.get(site): return float(AUTOCORR_RANGE_M[site])
    fits = [fit_variogram(xy, np.log1p(np.maximum(d[y].values, 0))) for y in TARGETS]
    rg = np.array([f["eff_range"] if (f["eff_range"] is not None and f["eff_range"] <= f["cutoff"]) else np.nan for f in fits], float)   # ranges beyond the lag cutoff are ignored
    r = np.nanmedian(rg) if np.isfinite(rg).any() else np.nan
    return 0.15 * dg if (not np.isfinite(r) or r > dg) else float(r)

def pred_cells(site, xy):                                                # map cells inside the plot hull + buffer
    R = RAST[site]; H, W = R["shape"]; t = R["transform"]; fct = max(1, int(np.sqrt(H * W / 40000)))
    rr, cc = np.meshgrid(np.arange(fct // 2, H, fct), np.arange(fct // 2, W, fct), indexing="ij"); rr, cc = rr.ravel(), cc.ravel()
    ok = R["footprint"][rr, cc]; rr, cc = rr[ok], cc[ok]
    lon, lat = t * (cc + 0.5, rr + 0.5); gx, gy = _t.transform(lon, lat)
    dg = np.hypot(np.ptp(xy[:, 0]), np.ptp(xy[:, 1])); hull = MultiPoint(xy).convex_hull.buffer(HULL_BUFFER * dg)
    m = shapely.contains_xy(hull, gx, gy); g = np.c_[gx[m], gy[m]]
    if len(g) > MAX_PRED_CELLS: g = g[np.random.default_rng(1).choice(len(g), MAX_PRED_CELLS, replace=False)]
    return g

QS = np.linspace(0.01, 0.99, 99)
def block_metrics(xy, fold, Gij, rng_m, Z=None):
    Gs = np.empty(len(xy)); En = np.empty(len(xy)) if Z is not None else None
    for f in np.unique(fold):
        te = fold == f
        Gs[te] = cKDTree(xy[~te]).query(xy[te])[0]
        if Z is not None: En[te] = cKDTree(Z[~te]).query(Z[te])[0]
    tab = pd.Series(fold).value_counts()
    return dict(n_folds=len(tab), n_min=int(tab.min()), n_max=int(tab.max()), n_cv=float(tab.std() / tab.mean()),
                W_rel=float(np.mean(np.abs(np.quantile(Gs, QS) - np.quantile(Gij, QS))) / np.median(Gij)),
                med_test_train=float(np.median(Gs)), med_pred=float(np.median(Gij)), med_ratio=float(np.median(Gs) / np.median(Gij)),
                leak=float(np.mean(Gs < rng_m)), env_nn=float(En.mean()) if Z is not None else np.nan)

def random_folds(n, k=None, seed=1):
    k = k or K_OUTER; return np.random.default_rng(seed).permutation(np.resize(np.arange(1, k + 1), n))

SITE_RNG = {s: site_range(s) for s in SITES}
SITE_XY  = {s: dat.loc[dat.Site == s, ["px", "py"]].values for s in SITES}
GIJ      = {s: nn_dist(pred_cells(s, SITE_XY[s]), SITE_XY[s]) for s in SITES}
print({s: f"{SITE_RNG[s] / 1000:.2f} km" for s in SITES}, "(median autocorrelation range)")

# ---------------------------------------------------------------- the search (BLOCK_MODE = "search")
def search_block_design(site, nseed=None):
    nseed = nseed or SEARCH_NSEED
    d = dat[dat.Site == site]; xy = SITE_XY[site]; n = len(d); dg = extent_diag(d); rg, Gij = SITE_RNG[site], GIJ[site]
    Z = d[[c for c in PRED_CONT if c in d]].values; Z = (Z - np.nanmean(Z, 0)) / np.where(np.nanstd(Z, 0) > 0, np.nanstd(Z, 0), 1); Z = np.nan_to_num(Z)
    env_ref = block_metrics(xy, random_folds(n), Gij, rg, Z)["env_nn"]
    sizes = sorted({round(max(1500, s) / 500) * 500 for s in [max(rg, 0.1 * dg) * m for m in (0.5, 1, 1.5, 2)] + [dg * f for f in (0.05, 0.08, 0.12, 0.18, 0.25)]})
    sizes = [s for s in sizes if s < 0.6 * dg]
    cands = [dict(id=f"{'hex' if hx else 'sq'}_{s / 1000:g}km", type="spatial", hexagon=hx, size=s) for s in sizes for hx in (False, True)]
    cands.append(dict(id="kmeans_xy", type="cluster"))
    runs = []
    for cd in cands:
        for sd in range(1, nseed + 1):
            try:
                if cd["type"] == "cluster": f = KMeans(K_OUTER, n_init=10, random_state=sd).fit_predict(xy) + 1
                else:
                    blk, _ = (hex_block_ids if cd["hexagon"] else square_block_ids)(xy, cd["size"]); f = assign_blocks(blk, K_OUTER, sd)
            except ValueError: continue
            m = block_metrics(xy, f, Gij, rg, Z); runs.append(dict(id=cd["id"], seed=sd, **m, env_infl=m["env_nn"] / env_ref))
    runs = pd.DataFrame(runs)
    agg = runs.groupby("id").agg(n_folds=("n_folds", "mean"), n_min=("n_min", "mean"), n_cv=("n_cv", "mean"), W_rel=("W_rel", "mean"),
                                 med_ratio=("med_ratio", "mean"), leak=("leak", "mean"), env_infl=("env_infl", "mean"), W_sd=("W_rel", "std")).reset_index()
    agg["eligible"] = (agg.n_folds == K_OUTER) & (agg.n_min >= SEARCH_MIN_NMIN)
    agg = agg.sort_values(["eligible", "W_rel"], ascending=[False, True]).reset_index(drop=True)
    e = agg[agg.eligible] if agg.eligible.any() else agg
    near = e[e.W_rel <= e.W_rel.min() + SEARCH_TOL]; best_id = near.loc[near.n_cv.idxmin(), "id"]
    ok = runs[(runs.id == best_id) & (runs.n_min >= SEARCH_MIN_NMIN)]; ok = ok if len(ok) else runs[runs.id == best_id]
    seed = int(ok.loc[ok.W_rel.idxmin(), "seed"]); cd = next(c for c in cands if c["id"] == best_id)
    return dict(design={k: v for k, v in cd.items() if k != "id"} | {"seed": seed}, summary=agg, runs=runs, tied=list(near.id))

BLOCK_SEARCH = None
if BLOCK_MODE == "search":
    BLOCK_SEARCH = cached(f"block_search_{CFG_TAG}", lambda: dict(Parallel(n_jobs=min(N_JOBS, 3))(delayed(lambda s: (s, search_block_design(s)))(s) for s in SITES)))
    for s in SITES: print(s, "->", design_label(BLOCK_SEARCH[s]["design"]), "| tied:", BLOCK_SEARCH[s]["tied"]); display(BLOCK_SEARCH[s]["summary"].head(6).round(3))
BLOCK_DESIGN = {s: BLOCK_SEARCH[s]["design"] for s in SITES} if BLOCK_SEARCH else BLOCK_FIXED

# ---------------------------------------------------------------- fold plans (ALL = union of the site folds)
PLANS = {s: make_plan_site(s, BLOCK_DESIGN[s]) for s in SITES}
PLAN_ALL = {s: PLANS[s]["plan"] for s in SITES}; PLAN_ALL = {"ALL": pd.concat(PLAN_ALL.values(), ignore_index=True)} | PLAN_ALL
PLAN_TAG = hashlib.md5(b"".join(pd.util.hash_pandas_object(PLAN_ALL[s], index=False).values.tobytes() for s in SCALES)).hexdigest()[:8]
print("fold-plan tag:", PLAN_TAG, "| source:", "R folds" if _RF_OK else "Python blockCV-like blocks")

def plan_data(sc, y):
    d = scale_data(sc); d = d[d[y].notna()].merge(PLAN_ALL[sc], on="plot_id", how="left")
    assert d.outer_.notna().all(), f"plots without a fold ({sc}/{y})"
    return d.reset_index(drop=True)

bd = []
for s in SITES:
    pl = PLANS[s]["plan"]; xy = SITE_XY[s]; mb = block_metrics(xy, pl.outer_.values, GIJ[s], SITE_RNG[s]); mr = block_metrics(xy, random_folds(len(xy)), GIJ[s], SITE_RNG[s])
    bd.append(dict(Scale=s, n=len(xy), Design=design_label(BLOCK_DESIGN[s]), Seed=BLOCK_DESIGN[s].get("seed"), N_blocks=pl.blk_.nunique(), Autocorr_range_km=SITE_RNG[s] / 1000,
                   Outer_n_min=mb["n_min"], Outer_n_max=mb["n_max"], Median_dist_map_km=mb["med_pred"] / 1000, Median_dist_blocks_km=mb["med_test_train"] / 1000,
                   Median_dist_random_km=mr["med_test_train"] / 1000, W_rel_blocks=mb["W_rel"], W_rel_random=mr["W_rel"], Leak_blocks=mb["leak"], Leak_random=mr["leak"]))
block_diag = pd.DataFrame(bd); display(block_diag.round(2))
fold_sizes = pd.DataFrame([dict(Scale=sc, Outer_fold=f, n_test=int((PLAN_ALL[sc].outer_ == f).sum()),
                                Inner_n_min=int(PLAN_ALL[sc].loc[PLAN_ALL[sc].outer_ != f, f"inner_o{f}"].value_counts().min()))
                           for sc in SCALES for f in range(1, K_OUTER + 1)])
display(fold_sizes.groupby("Scale").agg(n_test_min=("n_test", "min"), n_test_max=("n_test", "max"), Inner_n_min=("Inner_n_min", "min")))

In [ ]:
def block_polygons(site, plan_info, folds):
    """polygons of the blocks of a site (only if every block lies in a single fold)"""
    g, pl = plan_info["geom"], plan_info["plan"]
    if g["kind"] == "cluster": return []
    lab = pl.blk_.str.replace(f"^{site}_", "", regex=True).values; out = []
    for b in np.unique(lab):
        fs = np.unique(folds[lab == b])
        if len(fs) > 1: return []
        if g["kind"] == "square":
            a, c = map(int, b[1:].split("_")); x0, y0 = g["origin"] + np.array([a, c]) * g["size"]
            out.append((np.array([[x0, y0], [x0 + g["size"], y0], [x0 + g["size"], y0 + g["size"]], [x0, y0 + g["size"]]]), fs[0]))
        else:
            q, r = map(int, b[1:].split("_")); s_ = g["size"] / np.sqrt(3); cx, cy = s_ * np.sqrt(3) * (q + r / 2), s_ * 1.5 * r
            ang = np.radians(30 + 60 * np.arange(6)); out.append((np.c_[cx + s_ * np.cos(ang), cy + s_ * np.sin(ang)], fs[0]))
    return out

def fig_blocks():
    from matplotlib.patches import Polygon
    asp = [np.ptp(SITE_XY[s][:, 1]) / np.ptp(SITE_XY[s][:, 0]) for s in SITES]
    fig, axes = plt.subplots(3, 2, figsize=(7.2, 9.6), gridspec_kw=dict(height_ratios=[a + 0.12 for a in asp]))
    for i, s in enumerate(SITES):
        xy = SITE_XY[s] / 1000; pad = 0.07 * max(np.ptp(xy[:, 0]), np.ptp(xy[:, 1]))
        for j, scheme in enumerate(["random", "blocks"]):
            ax = axes[i, j]; fold = random_folds(len(xy)) if scheme == "random" else PLANS[s]["plan"].outer_.values
            if scheme == "blocks":
                for poly, f in block_polygons(s, PLANS[s], fold):
                    ax.add_patch(Polygon(poly / 1000, closed=True, fc=FOLD_COLS[f - 1], ec="#404040", lw=0.4, alpha=0.28))
            ax.scatter(xy[:, 0], xy[:, 1], c=[FOLD_COLS[f - 1] for f in fold], s=16, edgecolor="#262626", linewidth=0.3, zorder=3)
            ax.set_xlim(xy[:, 0].min() - pad, xy[:, 0].max() + pad); ax.set_ylim(xy[:, 1].min() - pad, xy[:, 1].max() + pad); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values(): sp.set_visible(True); sp.set_linewidth(0.6)
            md = block_metrics(SITE_XY[s], fold, GIJ[s], SITE_RNG[s])["med_test_train"] / 1000
            ax.set_title(f"{s}, {'random k-fold' if scheme == 'random' else 'spatial blocks'}: {md:.1f} km", loc="left", fontsize=10)
            ax.text(-0.02, 1.02, "abcdef"[2 * i + j], transform=ax.transAxes, fontsize=14, fontweight="bold", va="bottom", ha="right")
            bar = min([1, 2, 5, 10, 20, 50], key=lambda b: abs(b - np.ptp(xy[:, 0]) / 5)); x1, y1 = ax.get_xlim()[1], ax.get_ylim()[0]
            ax.plot([x1 - 0.04 * (x1 - ax.get_xlim()[0]) - bar, x1 - 0.04 * (x1 - ax.get_xlim()[0])], [y1 + 0.05 * (ax.get_ylim()[1] - y1)] * 2, "k-", lw=2, zorder=5)
            ax.text(x1 - 0.04 * (x1 - ax.get_xlim()[0]) - bar / 2, y1 + 0.07 * (ax.get_ylim()[1] - y1), f"{bar} km", ha="center", fontsize=8, zorder=5)
    h = [plt.Line2D([], [], marker="o", ls="", mfc=FOLD_COLS[k], mec="#262626", ms=7, label=str(k + 1)) for k in range(K_OUTER)]
    fig.legend(handles=h, title="Test fold", ncol=5, loc="lower center", frameon=False, bbox_to_anchor=(0.5, -0.03)); fig.tight_layout(rect=(0, 0.03, 1, 1))
    save_fig(fig, "Fig_spatial_blocks"); plt.show()
fig_blocks()

# 6. Models
* **RF** — `scikit-learn` (`max_samples = 0.632`, bootstrap); unordered categorical covariates are ordered by their mean response inside the training set (= `respect.unordered.factors = "order"` of ranger). **Variable selection** is VSURF-style but computed on the *inner spatial folds*: permutation importance on the held-out inner fold → thresholding (importance > its standard error) → interpretation (smallest nested model within 1 SE of the best inner-CV RMSE). **Optuna** (TPE sampler + median pruner) tunes `mtry`, `min_samples_leaf` on the inner spatial CV.
* **GAM** — additive model with **penalised cubic B-splines (P-splines, 2nd-order difference penalty)** for covariates with ≥ 6 distinct values (linear otherwise), dummy-coded categoricals, Gaussian/identity. Forward selection on the inner-CV RMSE (smoothing parameter by GCV); **Optuna** tunes the basis size `k ∈ {3,5,8}` and the smoothing parameter `λ`.
* **LMM** — `statsmodels` MixedLM; standardised continuous covariates; random intercept `Site` (ALL) or spatial `block` (sites); forward selection on AIC (ML, ΔAIC ≥ 2), final fit REML, **fixed effects only** for prediction.

In [ ]:
def rmse(o, p): return float(np.sqrt(np.mean((np.asarray(o) - np.asarray(p)) ** 2)))
def _clip(p): return np.maximum(p, 0) if CLIP_NEGATIVE else p

# ------------------------------------------------------------------ encoders
class RFEnc:
    """continuous as is; categorical -> rank of the class mean response (training set only); unseen class -> modal class"""
    def __init__(self, feats, cat): self.feats = list(feats); self.cat = [c for c in cat if c in self.feats]
    def fit(self, df, y):
        self.maps, self.modal = {}, {}
        for c in self.cat:
            g = df.groupby(c)[y].mean().sort_values(); self.maps[c] = {k: i for i, k in enumerate(g.index)}; self.modal[c] = df[c].mode().iloc[0]
        return self
    def transform(self, df):
        X = np.empty((len(df), len(self.feats)), np.float32)
        for j, f in enumerate(self.feats):
            if f in self.cat:
                m = self.maps[f]; X[:, j] = pd.Series(df[f].values).map(m).fillna(m[self.modal[f]]).values
            else: X[:, j] = df[f].values
        return X

def fit_rf(X, y, mtry, msl, ntree, seed, n_jobs=1):
    p = X.shape[1]
    return RandomForestRegressor(n_estimators=int(ntree), max_features=int(min(max(mtry, 1), p)), min_samples_leaf=int(msl), max_samples=0.632,
                                 bootstrap=True, n_jobs=n_jobs, random_state=int(seed)).fit(X, y)

def _inner_mats(df, y, feats, cat, inner):
    out = []
    for f in sorted(set(inner)):
        tr = inner != f; enc = RFEnc(feats, cat).fit(df[tr], y)
        out.append((enc.transform(df[tr]), df[y].values[tr], enc.transform(df[~tr]), df[y].values[~tr]))
    return out

def select_rf(df, y, cont, cat, inner, seed):
    feats = cont + cat; p = len(feats)
    if p < 2: return feats
    rng = np.random.default_rng(seed); mats = _inner_mats(df, y, feats, cat, inner); reps = 3; imp = np.zeros((len(mats), p))
    for i, (Xa, ya, Xb, yb) in enumerate(mats):                          # permutation importance on the held-out INNER spatial fold
        m = fit_rf(Xa, ya, max(1, p // 3), 3, 100, seed + i); base = np.mean((yb - m.predict(Xb)) ** 2); blocks = []
        for j in range(p):
            for _ in range(reps):
                Xp = Xb.copy(); Xp[:, j] = rng.permutation(Xp[:, j]); blocks.append(Xp)
        P = m.predict(np.vstack(blocks)).reshape(p, reps, -1)
        imp[i] = ((yb[None, None, :] - P) ** 2).mean(axis=2).mean(axis=1) - base
    mu, se = imp.mean(0), imp.std(0, ddof=1) / np.sqrt(len(mats))
    keep = [j for j in np.argsort(-mu) if mu[j] > 0 and mu[j] > se[j]] or [int(np.argmax(mu))]       # thresholding
    sc = []
    for m_ in range(1, len(keep) + 1):                                    # interpretation: nested models by decreasing importance
        fs = [feats[j] for j in keep[:m_]]; e = []
        for (Xa, ya, Xb, yb), (Xa2, _, Xb2, _) in zip(mats, _inner_mats(df, y, fs, cat, inner)):
            mm = fit_rf(Xa2, ya, max(1, m_ // 3), 3, 100, seed + 50); e.append(rmse(yb, mm.predict(Xb2)))
        sc.append((np.mean(e), np.std(e, ddof=1) / np.sqrt(len(e)) if len(e) > 1 else 0))
    b = int(np.argmin([s[0] for s in sc])); thr = sc[b][0] + sc[b][1]
    chosen = next(m_ for m_ in range(1, len(keep) + 1) if sc[m_ - 1][0] <= thr)                # smallest model within 1 SE
    return [feats[j] for j in keep[:chosen]]

def tune_rf(df, y, feats, cat, inner, seed, n_trials=None):
    n_trials = n_trials or N_TRIALS_RF; p = len(feats); mats = _inner_mats(df, y, feats, cat, inner); lo, hi = min(MTRY_RANGE[0], p), min(MTRY_RANGE[1], p)
    def obj(trial):
        mtry = trial.suggest_int("mtry", lo, hi); msl = trial.suggest_categorical("min_samples_leaf", MSL_CHOICES); errs = []
        for i, (Xa, ya, Xb, yb) in enumerate(mats):
            errs.append(rmse(yb, fit_rf(Xa, ya, mtry, msl, NUM_TREES_INNER, seed).predict(Xb))); trial.report(float(np.mean(errs)), i)
            if trial.should_prune(): raise optuna.TrialPruned()
        return float(np.mean(errs))
    st = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=seed, n_startup_trials=min(8, n_trials)),
                             pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1))
    st.optimize(obj, n_trials=n_trials, show_progress_bar=False)
    return dict(mtry=st.best_params["mtry"], min_node_size=st.best_params["min_samples_leaf"], k=np.nan, lam=np.nan, rmse=st.best_value)

def oob_perm_importance(m, X, y, seed=1):
    """OOB permutation importance (%IncMSE, as ranger / randomForest) computed on the forest's own out-of-bag samples"""
    from sklearn.ensemble._forest import _generate_sample_indices, _get_n_samples_bootstrap
    X = np.ascontiguousarray(X, np.float32); n, p = X.shape; rng = np.random.default_rng(seed); nb = _get_n_samples_bootstrap(n, m.max_samples)
    delta, cnt = np.zeros(p), 0; psum, pn = np.zeros(n), np.zeros(n)
    for t in m.estimators_:
        oob = np.setdiff1d(np.arange(n), _generate_sample_indices(t.random_state, n, nb))
        if len(oob) < 2: continue
        Xo, yo = X[oob], y[oob]; pb = t.tree_.predict(Xo).ravel(); eb = np.mean((yo - pb) ** 2); psum[oob] += pb; pn[oob] += 1; cnt += 1
        for j in range(p):
            Xp = Xo.copy(); Xp[:, j] = rng.permutation(Xp[:, j]); delta[j] += np.mean((yo - t.tree_.predict(Xp).ravel()) ** 2) - eb
    ok = pn > 0; oob_mse = np.mean((y[ok] - psum[ok] / pn[ok]) ** 2)
    return 100 * (delta / max(cnt, 1)) / max(oob_mse, 1e-8)

class RFModel:
    algo = "RF"
    def __init__(self, df, y, sel, cat, hp, seed, ntree=None, n_jobs=1, importance=False):
        self.sel, self.cat = list(sel), [c for c in cat if c in sel]; self.enc = RFEnc(self.sel, self.cat).fit(df, y)
        X = self.enc.transform(df); yy = df[y].values
        self.rf = fit_rf(X, yy, hp["mtry"], hp["min_node_size"], ntree or NUM_TREES, seed, n_jobs)
        self.imp = dict(zip(self.sel, oob_perm_importance(self.rf, X, yy, seed))) if importance else {}
    def predict(self, df): return _clip(self.rf.predict(self.enc.transform(df)))

# ------------------------------------------------------------------ GAM (P-splines)
class PSplineGAM:
    def __init__(self, cont, cat, k=5, eps=1e-4): self.cont, self.cat, self.k, self.eps = list(cont), list(cat), k, eps
    def init_basis(self, df):
        self.sp, self.lin, self.lv, self.modal, blocks, pen = {}, {}, {}, {}, [], []
        for v in self.cont:
            x = df[v].values.reshape(-1, 1); u = len(np.unique(x))
            if u >= 6:
                nk = max(min(self.k, u - 1) - 1, 2); st = SplineTransformer(n_knots=nk, degree=3, knots="quantile", include_bias=False, extrapolation="constant").fit(x)
                self.sp[v] = st; m = nk + 1; D = np.diff(np.eye(m), 2, axis=0); pen.append(D.T @ D)
            else:
                self.lin[v] = (float(x.mean()), float(x.std()) or 1.0); pen.append(np.zeros((1, 1)))
        for c in self.cat:
            self.known = getattr(self, "known", {}); self.known[c] = sorted(df[c].unique())
            self.lv[c] = self.known[c][1:]; self.modal[c] = df[c].mode().iloc[0]; pen.append(np.eye(len(self.lv[c])) * 1e-3)
        from scipy.linalg import block_diag
        self.P = block_diag(*pen) if pen else np.zeros((0, 0))
        return self
    def design(self, df):
        parts = []
        for v in self.cont:
            x = df[v].values.reshape(-1, 1)
            if v in self.sp: parts.append(self.sp[v].transform(x))
            else: mu, sd = self.lin[v]; parts.append((x - mu) / sd)
        for c in self.cat:
            x = df[c].values; x = np.where(np.isin(x, self.known[c]), x, self.modal[c])          # unseen class -> modal class
            parts.append(np.column_stack([(x == l).astype(float) for l in self.lv[c]]) if self.lv[c] else np.zeros((len(df), 0)))
        return np.hstack(parts) if parts else np.zeros((len(df), 0))
    def fit(self, df, y, lam="gcv", grid=np.logspace(-2, 3, 11)):
        yv = df[y].values.astype(float); X = self.design(df); n, p = X.shape
        if p == 0: self.beta, self.b0, self.lam = np.zeros(0), yv.mean(), np.nan; return self
        Xm = X.mean(0); Xc = X - Xm; yc = yv - yv.mean(); XtX, Xty = Xc.T @ Xc, Xc.T @ yc; I = np.eye(p) * self.eps
        def solve(l):
            A = XtX + l * self.P + I; return np.linalg.solve(A, Xty), A
        if lam == "gcv":
            best = None
            for l in grid:
                b, A = solve(l); edf = np.trace(np.linalg.solve(A, XtX)); rss = float(np.sum((yc - Xc @ b) ** 2)); g = n * rss / max(n - edf, 1e-6) ** 2
                if best is None or g < best[0]: best = (g, l, b)
            _, self.lam, self.beta = best
        else: self.lam = float(lam); self.beta, _ = solve(self.lam)
        self.b0 = float(yv.mean() - Xm @ self.beta); return self
    def refit(self, df, y):                                                  # same basis and lambda, new coefficients (bootstrap)
        g = PSplineGAM.__new__(PSplineGAM); g.__dict__.update(self.__dict__); return g.fit(df, y, lam=self.lam if np.isfinite(self.lam) else "gcv")
    def predict(self, df): return self.b0 + (self.design(df) @ self.beta if len(self.beta) else 0)

def cv_gam(df, y, cont_s, cat_s, inner, k, lam="gcv"):
    e = []
    for f in sorted(set(inner)):
        tr = inner != f; g = PSplineGAM(cont_s, cat_s, k).init_basis(df[tr]).fit(df[tr], y, lam=lam)
        e.append(rmse(df[y].values[~tr], _clip(g.predict(df[~tr]))))
    return float(np.mean(e))

def select_gam(df, y, cont, cat, inner):
    cand, sel = cont + cat, []; ev = lambda s: cv_gam(df, y, [v for v in s if v in cont], [v for v in s if v in cat], inner, K_GAM[len(K_GAM) // 2])
    best = ev(sel)
    while True:
        rem = [v for v in cand if v not in sel]
        if not rem: break
        sc = [ev(sel + [v]) for v in rem]; j = int(np.argmin(sc))
        if not np.isfinite(sc[j]) or (len(sel) > 0 and sc[j] > best * (1 - SEL_MIN_GAIN)): break
        sel.append(rem[j]); best = min(best, sc[j])
    return sel

def tune_gam(df, y, sel, cat, inner, seed, n_trials=None):
    n_trials = n_trials or N_TRIALS_GAM; cs, ks = [v for v in sel if v not in cat], [v for v in sel if v in cat]
    if not sel: return dict(mtry=np.nan, min_node_size=np.nan, k=K_GAM[1], lam=1.0, rmse=np.nan)
    def obj(trial): return cv_gam(df, y, cs, ks, inner, trial.suggest_categorical("k", K_GAM), trial.suggest_float("lam", 1e-2, 1e3, log=True))
    st = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=seed, n_startup_trials=min(8, n_trials))); st.optimize(obj, n_trials=n_trials, show_progress_bar=False)
    return dict(mtry=np.nan, min_node_size=np.nan, k=st.best_params["k"], lam=st.best_params["lam"], rmse=st.best_value)

class GAMModel:
    algo = "GAM"
    def __init__(self, df, y, sel, cat, hp):
        self.sel, self.cat = list(sel), [c for c in cat if c in sel]; self.cont = [v for v in self.sel if v not in self.cat]; self.y = y
        self.gam = PSplineGAM(self.cont, self.cat, int(hp["k"])).init_basis(df).fit(df, y, lam=float(hp["lam"]))
    def predict(self, df): return _clip(self.gam.predict(df))

# ------------------------------------------------------------------ LMM
class LMMDesign:
    def __init__(self, cont, cat): self.cont, self.cat = list(cont), list(cat)
    def fit(self, df):
        self.mu = {v: float(df[v].mean()) for v in self.cont}; self.sd = {v: float(df[v].std()) or 1.0 for v in self.cont}
        self.known = {c: sorted(df[c].unique()) for c in self.cat}; self.lv = {c: self.known[c][1:] for c in self.cat}
        self.modal = {c: df[c].mode().iloc[0] for c in self.cat}; return self
    def design(self, df):
        cols = [np.ones(len(df))] + [(df[v].values - self.mu[v]) / self.sd[v] for v in self.cont]
        for c in self.cat:
            x = np.where(np.isin(df[c].values, self.known[c]), df[c].values, self.modal[c]); cols += [(x == l).astype(float) for l in self.lv[c]]
        return np.column_stack(cols)

def mixed_fit(yv, X, g, reml):
    """random-intercept LMM; BFGS first (statsmodels' L-BFGS can return an infinite likelihood), OLS if the fit is degenerate or there are < 3 groups"""
    if pd.Series(g).nunique() >= 3:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                r = sm.MixedLM(yv, X, groups=g).fit(reml=reml, method=["bfgs", "lbfgs"], disp=False, maxiter=200)
            if np.isfinite(r.llf) and np.all(np.isfinite(r.fe_params)): return np.asarray(r.fe_params), float(-2 * r.llf + 2 * (X.shape[1] + 2))
        except Exception: pass
    r = sm.OLS(yv, X).fit(); return np.asarray(r.params), float(-2 * r.llf + 2 * (X.shape[1] + 1))

def fit_lmm(df, y, cont, cat, group_col, reml=True):
    des = LMMDesign(cont, cat).fit(df); beta, aic = mixed_fit(df[y].values.astype(float), des.design(df), df[group_col].values, reml)
    return dict(des=des, beta=beta, aic=aic)

def select_lmm(df, y, cont, cat, group_col):
    cand, sel = cont + cat, []
    aic = lambda s: fit_lmm(df, y, [v for v in s if v in cont], [v for v in s if v in cat], group_col, reml=False)["aic"]
    best = aic(sel)
    while True:
        rem = [v for v in cand if v not in sel]
        if not rem: break
        sc = [aic(sel + [v]) for v in rem]; j = int(np.argmin(sc))
        if not np.isfinite(sc[j]) or (len(sel) > 0 and sc[j] > best - LMM_DELTA_AIC): break
        sel.append(rem[j]); best = min(best, sc[j])
    return sel

class LMMModel:
    algo = "LMM"
    def __init__(self, df, y, sel, cat, group_col, reml=True):
        self.sel, self.cat = list(sel), [c for c in cat if c in sel]; self.cont = [v for v in self.sel if v not in self.cat]
        r = fit_lmm(df, y, self.cont, self.cat, group_col, reml); self.des, self.beta = r["des"], r["beta"]
    def predict(self, df): return _clip(self.des.design(df) @ self.beta)

# ------------------------------------------------------------------ generic wrappers
def select_vars(algo, df, y, cont, cat, inner, group_col, seed):
    return {"RF": lambda: select_rf(df, y, cont, cat, inner, seed), "GAM": lambda: select_gam(df, y, cont, cat, inner),
            "LMM": lambda: select_lmm(df, y, cont, cat, group_col)}[algo]()
def tune_hp(algo, df, y, sel, cat, inner, seed):
    if algo == "RF": return tune_rf(df, y, sel, cat, inner, seed)
    if algo == "GAM": return tune_gam(df, y, sel, cat, inner, seed)
    return dict(mtry=np.nan, min_node_size=np.nan, k=np.nan, lam=np.nan, rmse=np.nan)
def fit_algo(algo, df, y, sel, cat, hp, group_col, seed=1, importance=False, n_jobs=1, ntree=None):
    if algo == "RF": return RFModel(df, y, sel, cat, hp, seed, ntree=ntree, n_jobs=n_jobs, importance=importance)
    if algo == "GAM": return GAMModel(df, y, sel, cat, hp)
    return LMMModel(df, y, sel, cat, group_col)

# ------------------------------------------------------------------ one nested-CV task = (scale, pool, outer fold)
def cv_task(d, y, cont, cat, f, group_col, seed):
    warnings.simplefilter("ignore")
    tr, te = d[d.outer_ != f].reset_index(drop=True), d[d.outer_ == f].reset_index(drop=True)
    inner = tr[f"inner_o{f}"].values.astype(int); assert (inner > 0).all()
    preds, sels, hps = pd.DataFrame(dict(plot_id=te.plot_id, fold=f, obs=te[y].values)), [], []
    allp = cont + cat
    for a in ALGOS:
        try:
            sel = select_vars(a, tr, y, cont, cat, inner, group_col, seed + f)                  # inside the outer training set only
            hp = tune_hp(a, tr, y, sel, cat, inner, seed + f)
            mdl = fit_algo(a, tr, y, sel, cat, hp, group_col, seed + f, importance=(a == "RF"))
            preds[a] = mdl.predict(te)
            sels += [dict(fold=f, algo=a, Predictor=v, selected=v in sel, pct_IncMSE=mdl.imp.get(v, 0.0) if a == "RF" else np.nan) for v in allp]
            hps.append(dict(fold=f, algo=a, **hp))
        except Exception as e:
            preds[a] = np.nan; print(f"  fold {f} / {a} failed: {type(e).__name__}: {e}")
    return dict(pred=preds, sel=pd.DataFrame(sels), hp=pd.DataFrame(hps))

# 7. Nested spatial block cross-validation
Every task = (scale, pool, outer fold). Inside the outer training set only: variable selection → Optuna tuning (inner spatial folds) → refit → prediction of the outer test fold. RF, GAM and LMM use **the same folds**. Tasks run in parallel and are cached one by one (re-running the cell resumes an interrupted run).

In [ ]:
def _run_task(sc, y, f, d, cont, cat, gc, tag):
    warnings.simplefilter("ignore")
    fn = CACHE / f"cv_{sc}_{y}_{f}_{tag}.pkl"
    if fn.exists():
        with open(fn, "rb") as fh: return pickle.load(fh)
    r = cv_task(d, y, cont, cat, f, gc, SEED + 11)
    with open(fn, "wb") as fh: pickle.dump(r, fh)
    return r

jobs = []
for sc in SCALES:
    ps = PRED_SETS[sc]; gc = "Site" if sc == "ALL" else "blk_"
    for y in TARGETS:
        d = plan_data(sc, y)
        for f in range(1, K_OUTER + 1): jobs.append((sc, y, f, d, ps["cont"], ps["cat"], gc))
jobs.sort(key=lambda j: -len(j[3]))                                       # biggest tasks first (better load balance)
t0 = time.time(); tag = f"{PLAN_TAG}_{CFG_TAG}"
print(f"{len(jobs)} tasks on {N_JOBS} workers ...")
res = Parallel(n_jobs=N_JOBS, verbose=5)(delayed(_run_task)(sc, y, f, d, c, k, gc, tag) for sc, y, f, d, c, k, gc in jobs)
print(f"nested CV done in {(time.time() - t0) / 60:.1f} min")

pred_l, sel_l, hp_l = [], [], []
for (sc, y, f, *_), r in zip(jobs, res):
    pred_l.append(r["pred"].assign(Scale=sc, Pool=y)); sel_l.append(r["sel"].assign(Scale=sc, Pool=y)); hp_l.append(r["hp"].assign(Scale=sc, Pool=y))
cv_wide = pd.concat(pred_l, ignore_index=True); cv_sel = pd.concat(sel_l, ignore_index=True); cv_hp = pd.concat(hp_l, ignore_index=True)
cv_long = cv_wide.melt(id_vars=["Scale", "Pool", "plot_id", "fold", "obs"], value_vars=ALGOS, var_name="Model", value_name="pred")
print("failed predictions:", int(cv_long.pred.isna().sum()), "of", len(cv_long))
display(cv_long.dropna().groupby(["Scale", "Model"]).fold.nunique().unstack().rename_axis("outer folds used"))

# 8. Accuracy, model comparison and best algorithm
Per-fold R², RMSE, nRMSE (% of the mean), MAE and RPD (= SD obs / RMSE); mean ± SD and t-based 95 % CI over the outer folds; paired Wilcoxon tests (Holm-adjusted per scale and metric); the best algorithm has the lowest rank sum of mean MAE and RMSE.

In [ ]:
def fold_metrics(o, p):
    o, p = np.asarray(o, float), np.asarray(p, float); ok = np.isfinite(o) & np.isfinite(p); o, p = o[ok], p[ok]; n = len(o)
    if n < 3: return dict(n=n, R2=np.nan, RMSE=np.nan, nRMSE=np.nan, MAE=np.nan, RPD=np.nan)
    r = rmse(o, p); sst = np.sum((o - o.mean()) ** 2)
    return dict(n=n, R2=1 - np.sum((p - o) ** 2) / sst if sst > 0 else np.nan, RMSE=r, nRMSE=100 * r / o.mean() if o.mean() > 0 else np.nan,
                MAE=np.mean(np.abs(p - o)), RPD=o.std(ddof=1) / r if r > 0 and o.std(ddof=1) > 0 else np.nan)
METRICS = ["R2", "RMSE", "nRMSE", "MAE", "RPD"]
cv_fold = pd.DataFrame([dict(Scale=sc, Pool=y, Model=m, fold=f, **fold_metrics(g.obs, g.pred))
                        for (sc, y, m, f), g in cv_long.groupby(["Scale", "Pool", "Model", "fold"])])
def summ_ci(x):
    x = np.asarray(x, float); x = x[np.isfinite(x)]; K = len(x)
    if K == 0: return pd.Series(dict(mean=np.nan, sd=np.nan, lo=np.nan, hi=np.nan, K=0))
    m = x.mean(); s = x.std(ddof=1) if K > 1 else np.nan; h = stats.t.ppf(0.975, K - 1) * s / np.sqrt(K) if K > 1 else np.nan
    return pd.Series(dict(mean=m, sd=s, lo=m - h, hi=m + h, K=K))
cv_summary = (cv_fold.melt(id_vars=["Scale", "Pool", "Model", "fold"], value_vars=METRICS, var_name="Metric")
              .groupby(["Scale", "Pool", "Model", "Metric"]).value.apply(summ_ci).unstack().reset_index())
ov = [dict(Scale=sc, Pool=y, Model=m, **{k: v for k, v in fold_metrics(g.obs, g.pred).items() if k != "n"}) for (sc, y, m), g in cv_long.groupby(["Scale", "Pool", "Model"])]
overall = pd.DataFrame(ov)                                                  # pooled out-of-fold metrics

pairs = [("RF", "GAM"), ("RF", "LMM"), ("GAM", "LMM")]; trows = []
for (sc, y, met), g in cv_fold[cv_fold.Model.isin(ALGOS)].melt(id_vars=["Scale", "Pool", "Model", "fold"], value_vars=["MAE", "RMSE"], var_name="Metric").groupby(["Scale", "Pool", "Metric"]):
    w = g.pivot(index="fold", columns="Model", values="value")
    for a, b in pairs:
        ok = w[[a, b]].dropna(); dif = ok[a] - ok[b]
        try: st = stats.wilcoxon(ok[a], ok[b]); V, p = st.statistic, st.pvalue
        except Exception: V, p = np.nan, np.nan
        trows.append(dict(Scale=sc, Pool=y, Metric=met, Comparison=f"{a} vs {b}", K=len(ok), Median_diff=dif.median(), V=V, p=p, Better=a if dif.median() < 0 else b))
model_tests = pd.DataFrame(trows)
model_tests["p_Holm"] = model_tests.groupby(["Scale", "Metric"]).p.transform(lambda p: multipletests(p.fillna(1), method="holm")[1])
tab7 = (cv_summary[cv_summary.Metric == "MAE"].pivot(index=["Scale", "Pool"], columns="Model", values="mean").add_prefix("MAE_").reset_index()
        .merge(model_tests[(model_tests.Metric == "MAE") & (model_tests.Comparison == "RF vs GAM")][["Scale", "Pool", "p_Holm"]].rename(columns={"p_Holm": "p_RF_vs_GAM"}), on=["Scale", "Pool"], how="left")
        .merge(model_tests[(model_tests.Metric == "MAE") & (model_tests.Comparison == "RF vs LMM")][["Scale", "Pool", "p_Holm"]].rename(columns={"p_Holm": "p_RF_vs_LMM"}), on=["Scale", "Pool"], how="left"))
tab7["Pool"] = tab7.Pool.map(TARGETS)

bt = cv_summary[cv_summary.Metric.isin(["MAE", "RMSE"])].pivot(index=["Scale", "Pool", "Model"], columns="Metric", values="mean").dropna().reset_index()
bt["rk"] = bt.groupby(["Scale", "Pool"]).MAE.rank() + bt.groupby(["Scale", "Pool"]).RMSE.rank()
_rows = []
for (sc, y), g in bt.groupby(["Scale", "Pool"]):
    b0 = g.sort_values(["rk", "MAE"]).iloc[0]
    _rows.append(dict(Scale=sc, Pool=y, Best=b0.Model, MAE_best=b0.MAE, RMSE_best=b0.RMSE, MAE_RMSE_agree=bool(g.MAE.idxmin() == g.RMSE.idxmin())))
best_tab = pd.DataFrame(_rows)

fmt = lambda g, m: f"{g.loc[g.Metric == m, 'mean'].iloc[0]:.2f} ± {g.loc[g.Metric == m, 'sd'].iloc[0]:.2f}"
acc = pd.DataFrame([dict(Scale=sc, Pool=TARGETS[y], Model=m, MAE=fmt(g, "MAE"), R2=fmt(g, "R2"), RMSE=fmt(g, "RMSE"), nRMSE=fmt(g, "nRMSE"), RPD=fmt(g, "RPD"))
                    for (sc, y, m), g in cv_summary.groupby(["Scale", "Pool", "Model"])])
display(acc); display(tab7.round(3)); display(best_tab.assign(Pool=best_tab.Pool.map(TARGETS)).round(2))

In [ ]:
def fig_cv():
    sub = cv_long.merge(best_tab[["Scale", "Pool", "Best"]], left_on=["Scale", "Pool", "Model"], right_on=["Scale", "Pool", "Best"]).dropna(subset=["pred"])
    pools = list(TARGETS); fig, axes = plt.subplots(len(SCALES), len(pools), figsize=(2.6 * len(pools), 2.6 * len(SCALES)), squeeze=False)
    for i, sc in enumerate(SCALES):
        for j, y in enumerate(pools):
            ax = axes[i, j]; g = sub[(sub.Scale == sc) & (sub.Pool == y)]
            if len(g):
                ax.scatter(g.pred, g.obs, s=9, c=PAL_SCALE[sc], alpha=0.65, edgecolor="none"); lim = [0, max(g.obs.max(), g.pred.max()) * 1.05]
                ax.plot(lim, lim, "k--", lw=0.7); ax.set_xlim(lim); ax.set_ylim(lim); m = fold_metrics(g.obs, g.pred)
                ax.text(0.04, 0.96, f"RMSE {m['RMSE']:.1f}\nMAE {m['MAE']:.1f}\nR² {m['R2']:.2f}\n({g.Model.iloc[0]})", transform=ax.transAxes, va="top", fontsize=7)
            if i == 0: ax.set_title(TARGETS[y], fontsize=9)
            if j == 0: ax.set_ylabel(f"{sc}\nObserved")
            if i == len(SCALES) - 1: ax.set_xlabel("Predicted")
    fig.tight_layout(); save_fig(fig, "Fig_obs_vs_pred_nestedCV"); plt.show()
fig_cv()

def fig_models():
    fig, axes = plt.subplots(2, len(SCALES), figsize=(3.4 * len(SCALES), 6.2), sharey="row")
    for r, met in enumerate(["MAE", "R2"]):
        for c, sc in enumerate(SCALES):
            ax = axes[r, c]
            for k, m in enumerate(ALGOS):
                s = cv_summary[(cv_summary.Scale == sc) & (cv_summary.Metric == met) & (cv_summary.Model == m)]
                x = np.arange(len(TARGETS)) + (k - 1) * 0.22
                ax.errorbar(x, s.set_index("Pool").reindex(list(TARGETS))["mean"], yerr=[(s.set_index("Pool").reindex(list(TARGETS))["mean"] - s.set_index("Pool").reindex(list(TARGETS))["lo"]).fillna(0),
                            (s.set_index("Pool").reindex(list(TARGETS))["hi"] - s.set_index("Pool").reindex(list(TARGETS))["mean"]).fillna(0)], fmt="o", ms=4, color=PAL_MOD[m], label=m, lw=1)
            ax.set_xticks(range(len(TARGETS))); ax.set_xticklabels(list(TARGETS), rotation=60, ha="right", fontsize=8)
            if r == 0: ax.set_title(sc)
            if c == 0: ax.set_ylabel("MAE (Mg C/ha)" if met == "MAE" else "R²")
            if met == "R2": ax.axhline(0, color="grey", lw=0.5)
    axes[0, 0].legend(frameon=False, fontsize=8); fig.tight_layout(); save_fig(fig, "Fig_model_comparison_metrics"); plt.show()
fig_models()

# 9. Final models, importance, partial dependence and bootstrap maps
For every scale × pool the **best algorithm** is used. Final selection and tuning are repeated on all plots (5 inner spatial folds from the plan); the model is refitted on all plots; **B = 100 bootstrap models** (plots resampled with replacement **within spatial blocks**) give the per-pixel mean, SD and 2.5–97.5 % interval and the site totals (Tg C).
* GAM / LMM bootstraps keep the basis (knots, standardisation) of the full model and re-estimate the coefficients, so a whole map chunk is predicted for all B models with **one matrix product**.
* RF bootstraps use `NUM_TREES_BOOT` trees per model (B × 50 trees); Monte-Carlo noise from the small forests is negligible against between-bootstrap spread.
* Maps are predicted at `MAP_AGG_FACT` × the native resolution (continuous layers averaged, categorical layers = most frequent class); set `MAP_AGG_FACT = 1` for the native grid.

In [ ]:
def imp_summary(sc, y, algo):
    s = cv_sel[(cv_sel.Scale == sc) & (cv_sel.Pool == y)]; K = s.fold.nunique(); s = s[s.algo == algo]
    g = s.groupby("Predictor").agg(N_folds_selected=("selected", "sum"), pct_IncMSE_mean=("pct_IncMSE", "mean"), pct_IncMSE_sd=("pct_IncMSE", "std")).reset_index()
    g["Freq"] = g.N_folds_selected / K
    g["Main"] = (g.Freq >= SEL_FREQ_MIN) & ((g.pct_IncMSE_mean >= IMP_REL_MIN * g.pct_IncMSE_mean.max()) if algo == "RF" else True)
    return g.sort_values(["Freq", "pct_IncMSE_mean"], ascending=False).reset_index(drop=True)

def pd_effect(model, d, v):
    if v not in model.sel: return dict(Predictor=v, Direction=None, Spearman_PD=np.nan, Effect_range=np.nan, Detail="not in the final model")
    if v in PRED_CAT:
        lv = sorted(d[v].unique()); pdv = np.array([model.predict(d.assign(**{v: l})).mean() for l in lv])
        return dict(Predictor=v, Direction="categorical", Spearman_PD=np.nan, Effect_range=float(np.ptp(pdv)), Detail="; ".join(f"{int(l)}: {p:.1f}" for l, p in zip(lv, pdv)))
    g = np.unique(np.quantile(d[v], np.linspace(0.05, 0.95, 20))); pdv = np.array([model.predict(d.assign(**{v: x})).mean() for x in g])
    rho = stats.spearmanr(g, pdv)[0] if len(g) > 2 and np.ptp(pdv) > 0 else np.nan
    dr = "non-monotonic / flat" if (not np.isfinite(rho) or abs(rho) < 0.5) else ("positive" if rho > 0 else "negative")
    return dict(Predictor=v, Direction=dr, Spearman_PD=rho, Effect_range=float(np.ptp(pdv)), Detail=f"PD {pdv[0]:.1f} (P5 = {g[0]:.2f}) to {pdv[-1]:.1f} (P95 = {g[-1]:.2f})")

def boot_index(d, B, seed):                                              # resample plots with replacement WITHIN each spatial block
    rng = np.random.default_rng(seed); groups = [np.flatnonzero(m) for m in (d.blk_.values == b for b in d.blk_.unique())]
    return [np.concatenate([g if len(g) == 1 else rng.choice(g, len(g), replace=True) for g in groups]) for _ in range(B)]

def lmm_beta(des, dfb, y, gc):                                           # coefficients of the full-model design re-estimated on a bootstrap sample
    return mixed_fit(dfb[y].values.astype(float), des.design(dfb), dfb[gc].values, True)[0]

class LinearEns:                                                         # GAM / LMM: one matrix product predicts all B models
    def __init__(self, design_fn, betas, b0, sel, cat): self.design_fn, self.betas, self.b0, self.sel, self.cat = design_fn, betas, b0, list(sel), [c for c in cat if c in sel]
    def predict_all(self, df):
        P = self.design_fn(df) @ self.betas.T + self.b0[None, :]; return np.maximum(P, 0) if CLIP_NEGATIVE else P
class RFEns:
    def __init__(self, models, sel, cat): self.models, self.sel, self.cat = models, list(sel), [c for c in cat if c in sel]
    def predict_all(self, df): return np.column_stack([m.predict(df) for m in self.models])

def model_terms(obj):
    if obj.algo == "LMM":
        names = ["Intercept"] + obj.cont + [f"{c}={int(l)}" for c in obj.cat for l in obj.des.lv[c]]
        return pd.DataFrame(dict(Term=names, Estimate=obj.beta))
    if obj.algo == "GAM": return pd.DataFrame(dict(Term=obj.sel, k=obj.gam.k, lam=obj.gam.lam if len(obj.sel) else np.nan))
    return pd.DataFrame()

def final_one(sc, y, best, d, cont, cat, gc, isum, seed_ix):
    inner = d.inner_full.values.astype(int)
    sel = select_vars(best, d, y, cont, cat, inner, gc, 99); hp = tune_hp(best, d, y, sel, cat, inner, 99)
    obj = fit_algo(best, d, y, sel, cat, hp, gc, 123, importance=False, ntree=NUM_TREES)
    main = list(isum.Predictor[isum.Main])
    pdm = fit_algo("RF", d, y, list(dict.fromkeys(sel + main)), cat, hp, gc, 123) if best == "RF" else obj
    pdt = pd.DataFrame([pd_effect(pdm, d, v) for v in dict.fromkeys(sel + main) if v in cont + cat])
    isum = isum.merge(pdt, on="Predictor", how="left") if len(pdt) else isum
    bi = boot_index(d, B_BOOT, 1000 + seed_ix); ens = None
    if best == "RF":
        ms = [fit_algo("RF", d.iloc[b].reset_index(drop=True), y, sel, cat, hp, gc, i, ntree=NUM_TREES_BOOT) for i, b in enumerate(bi)]; ens = RFEns(ms, sel, cat)
    elif best == "GAM":
        gs = [obj.gam.refit(d.iloc[b].reset_index(drop=True), y) for b in bi]; ens = LinearEns(obj.gam.design, np.vstack([g.beta if len(g.beta) else np.zeros(0) for g in gs]), np.array([g.b0 for g in gs]), sel, cat)
    else:
        bs = [lmm_beta(obj.des, d.iloc[b].reset_index(drop=True), y, gc) for b in bi]; ens = LinearEns(obj.des.design, np.vstack(bs), np.zeros(len(bs)), sel, cat)
    return dict(best=best, sel=sel, hp=hp, n=len(d), importance=isum, terms=model_terms(obj), ens=ens, B_ok=len(bi))

def _final_task(sc, y, best, d, cont, cat, gc, isum, seed_ix, tag):
    warnings.simplefilter("ignore")
    fn = CACHE / f"final_{sc}_{y}_{best}_{tag}.pkl"
    if fn.exists():
        with open(fn, "rb") as fh: return pickle.load(fh)
    r = final_one(sc, y, best, d, cont, cat, gc, isum, seed_ix)
    with open(fn, "wb") as fh: pickle.dump(r, fh)
    return r

fjobs = []
for sc in SCALES:
    for y in TARGETS:
        best = best_tab.loc[(best_tab.Scale == sc) & (best_tab.Pool == y), "Best"].iloc[0]; ps = PRED_SETS[sc]
        fjobs.append((sc, y, best, plan_data(sc, y), ps["cont"], ps["cat"], "Site" if sc == "ALL" else "blk_", imp_summary(sc, y, best), SCALES.index(sc) * 10 + list(TARGETS).index(y), tag))
fjobs.sort(key=lambda j: -len(j[3])); t0 = time.time()
fres = Parallel(n_jobs=N_JOBS, verbose=5)(delayed(_final_task)(*j) for j in fjobs)
FINAL = {(j[0], j[1]): r for j, r in zip(fjobs, fres)}
print(f"final models + {B_BOOT} bootstraps: {(time.time() - t0) / 60:.1f} min")
hp_final = pd.DataFrame([dict(Scale=sc, Pool=TARGETS[y], n=o["n"], Algorithm=o["best"], Selected=", ".join(o["sel"]), mtry=o["hp"].get("mtry"), min_node_size=o["hp"].get("min_node_size"),
                              k=o["hp"].get("k"), lam=o["hp"].get("lam"), Inner_CV_RMSE=o["hp"].get("rmse"), Bootstrap_models=o["B_ok"]) for (sc, y), o in FINAL.items()])
display(hp_final.sort_values(["Scale", "Pool"]).round(3))

In [ ]:
imp_tab = pd.concat([o["importance"].assign(Scale=sc, Pool=TARGETS[y], Algorithm=o["best"]) for (sc, y), o in FINAL.items()], ignore_index=True)
def top_text(g, n=4):
    out = []
    for _, r in g[g.Main].head(n).iterrows():
        dsc = f"{r.pct_IncMSE_mean:.1f} ± {0 if pd.isna(r.pct_IncMSE_sd) else r.pct_IncMSE_sd:.1f}" if r.Algorithm == "RF" else f"selected in {100 * r.Freq:.0f}% of folds"
        out.append(f"{r.Predictor} ({dsc}; {r.Direction if isinstance(r.Direction, str) else 'NA'})")
    return out + [""] * (n - len(out))
main_tab = pd.DataFrame([dict(Scale=sc, Pool=p, Algorithm=g.Algorithm.iloc[0], **{f"Covariate {i + 1}": t for i, t in enumerate(top_text(g))}) for (sc, p), g in imp_tab.groupby(["Scale", "Pool"])])
display(main_tab)

def fig_importance():
    sub = imp_tab[(imp_tab.Algorithm == "RF") & imp_tab.Main]; sc_list = SCALES
    fig, axes = plt.subplots(len(sc_list), len(TARGETS), figsize=(2.9 * len(TARGETS), 2.3 * len(sc_list)), squeeze=False)
    for i, sc in enumerate(sc_list):
        for j, y in enumerate(TARGETS):
            ax = axes[i, j]; g = sub[(sub.Scale == sc) & (sub.Pool == TARGETS[y])].sort_values("pct_IncMSE_mean")
            if len(g): ax.barh(g.Predictor, g.pct_IncMSE_mean, xerr=g.pct_IncMSE_sd.fillna(0), color=PAL_SCALE[sc], height=0.7, error_kw=dict(lw=0.6))
            else: ax.text(0.5, 0.5, "best model is not RF", ha="center", va="center", fontsize=7, transform=ax.transAxes); ax.set_xticks([]); ax.set_yticks([])
            if i == 0: ax.set_title(TARGETS[y], fontsize=9)
            if j == 0: ax.set_ylabel(sc)
            ax.tick_params(labelsize=7)
    fig.supxlabel("%IncMSE (OOB permutation, mean ± SD over outer folds)", fontsize=9); fig.tight_layout(); save_fig(fig, "Fig_variable_importance"); plt.show()
fig_importance()

## Bootstrap maps and site totals

In [ ]:
def aggregate_layer(a, fact, categorical):
    if fact == 1: return a
    H, W = a.shape; Hp, Wp = -(-H // fact) * fact, -(-W // fact) * fact; b = np.full((Hp, Wp), np.nan, np.float32); b[:H, :W] = a
    b = b.reshape(Hp // fact, fact, Wp // fact, fact)
    if not categorical: return np.nanmean(b, axis=(1, 3)).astype(np.float32)
    cl = np.unique(a[~np.isnan(a)]); cnt = np.stack([(b == c).sum(axis=(1, 3)) for c in cl]); out = cl[cnt.argmax(0)].astype(np.float32); out[cnt.sum(0) == 0] = np.nan
    return out

MAPPACK = {}
for s in SITES:
    R = RAST[s]; lay = {c: aggregate_layer(a, MAP_AGG_FACT, c in PRED_CAT) for c, a in R["layers"].items()}
    t = R["transform"] * Affine.scale(MAP_AGG_FACT); H, W = next(iter(lay.values())).shape
    lat = t.f + t.e * (np.arange(H) + 0.5); dlat, dlon = np.radians(abs(t.e)), np.radians(abs(t.a)); Re = 6371.0088
    area_ha_row = (Re ** 2 * dlon * (np.sin(np.radians(lat) + dlat / 2) - np.sin(np.radians(lat) - dlat / 2)) * 100).astype(np.float64)   # km2 -> ha
    MAPPACK[s] = dict(layers=lay, transform=t, crs=R["crs"], shape=(H, W), area_ha_row=np.abs(area_ha_row),
                      levels={c: np.unique(dat[c].dropna()) for c in PRED_CAT})
    print(s, "map grid", (H, W), f"{int(np.isfinite(lay['Sin_aspect']).sum()):,} cells")

def predict_map(pack, ens, tag, site):
    H, W = pack["shape"]; lay = pack["layers"]; B = (ens.betas.shape[0] if hasattr(ens, "betas") else len(ens.models))
    foot = np.isfinite(lay["Sin_aspect"]); m_, s_, lo_, hi_ = (np.full((H, W), np.nan, np.float32) for _ in range(4))
    tot = np.zeros(B); n_ok = n_mask = 0; area = 0.0; rows = max(1, CHUNK_CELLS // W)
    for r0 in range(0, H, rows):
        r1 = min(r0 + rows, H); fm = foot[r0:r1]
        v = pd.DataFrame({c: lay[c][r0:r1].ravel() for c in ens.sel}); ok = fm.ravel() & v.notna().all(axis=1).values
        for c in ens.cat: ok &= np.isin(v[c].values, pack["levels"][c])                      # class never seen in the plots -> masked
        n_mask += int(fm.sum() - ok.sum())
        if not ok.any(): continue
        P = ens.predict_all(v[ok].reset_index(drop=True)); a_ha = np.repeat(pack["area_ha_row"][r0:r1], W)[ok]
        q = np.quantile(P, [0.025, 0.975], axis=1); idx = np.flatnonzero(ok)
        for arr, val in ((m_, P.mean(1)), (s_, P.std(1, ddof=1)), (lo_, q[0]), (hi_, q[1])): arr[r0:r1].ravel()[idx] = val
        tot += (P * a_ha[:, None]).sum(0) * 1e-6; n_ok += int(ok.sum()); area += float(a_ha.sum())
    out = MAPS / f"{tag}.tif"
    with rasterio.open(out, "w", driver="GTiff", height=H, width=W, count=4, dtype="float32", crs=pack["crs"], transform=pack["transform"], nodata=np.nan, compress="lzw") as dst:
        for i, (arr, nm) in enumerate(zip((m_, s_, lo_, hi_), ["mean", "sd", "q025", "q975"]), 1): dst.write(arr, i); dst.set_band_description(i, nm)
    return dict(file=str(out), totals=tot, n_valid=n_ok, n_masked=n_mask, area_ha=area)

def _map_task(sc, y, st, ens, pack, tag):
    warnings.simplefilter("ignore")
    fn = CACHE / f"map_{sc}_{y}_{st}_{tag}.pkl"
    if fn.exists() and (MAPS / f"{sc}_{y}_{st}.tif").exists():
        with open(fn, "rb") as fh: return pickle.load(fh)
    r = predict_map(pack, ens, f"{sc}_{y}_{st}", st)
    with open(fn, "wb") as fh: pickle.dump(r, fh)
    return r

mjobs = [(sc, y, st, FINAL[(sc, y)]["ens"], MAPPACK[st], f"{tag}_{MAP_AGG_FACT}") for (sc, y) in FINAL for st in (SITES if sc == "ALL" else [sc])]
mjobs.sort(key=lambda j: -(MAPPACK[j[2]]["shape"][0] * MAPPACK[j[2]]["shape"][1])); t0 = time.time()
mres = Parallel(n_jobs=N_JOBS, verbose=5)(delayed(_map_task)(*j) for j in mjobs)
print(f"maps: {(time.time() - t0) / 60:.1f} min")
tot_rows = []
for (sc, y, st, *_), r in zip(mjobs, mres):
    o = FINAL[(sc, y)]; t = r["totals"]
    tot_rows.append(dict(Scale=sc, Pool=TARGETS[y], Site=st, Algorithm=o["best"], B=o["B_ok"], Pixels=r["n_valid"], Masked_pixels=r["n_masked"],
                         Masked_pct=100 * r["n_masked"] / max(r["n_valid"] + r["n_masked"], 1), Area_ha=r["area_ha"], Total_TgC_mean=t.mean(), Total_TgC_sd=t.std(ddof=1),
                         Total_TgC_q025=np.quantile(t, 0.025), Total_TgC_q975=np.quantile(t, 0.975)))
totals = pd.DataFrame(tot_rows).sort_values(["Scale", "Pool", "Site"]).reset_index(drop=True); display(totals.round(3))

def fig_maps(sc="ALL", y="TotC"):
    fig, axes = plt.subplots(len(SITES), 3, figsize=(10, 3.3 * len(SITES)), squeeze=False)
    for i, st in enumerate(SITES):
        with rasterio.open(MAPS / f"{sc}_{y}_{st}.tif") as src:
            m, s_, lo, hi = (src.read(k) for k in (1, 2, 3, 4)); t = src.transform
        ys, xs = np.where(np.isfinite(m)); sl = (slice(ys.min(), ys.max() + 1), slice(xs.min(), xs.max() + 1))              # crop to the valid area
        ext = (t.c + xs.min() * t.a, t.c + (xs.max() + 1) * t.a, t.f + (ys.max() + 1) * t.e, t.f + ys.min() * t.e); m, s_, lo, hi = m[sl], s_[sl], lo[sl], hi[sl]
        for j, (arr, ttl, cm) in enumerate([(m, f"{TARGETS[y]} (Mg C/ha)", "viridis"), (s_, "Bootstrap SD", "magma"), (hi - lo, "95 % interval width", "magma")]):
            ax = axes[i, j]; im = ax.imshow(arr, extent=ext, cmap=cm, vmin=np.nanpercentile(arr, 2), vmax=np.nanpercentile(arr, 98)); ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values(): sp.set_visible(True)
            ax.set_title(f"{st} · {ttl}", loc="left", fontsize=9); fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02).ax.tick_params(labelsize=7)
    fig.tight_layout(); save_fig(fig, f"Map_{sc}_{y}", dpi=300); plt.show()
fig_maps("ALL", "TotC" if "TotC" in TARGETS else next(iter(TARGETS)))

# 10. Residual spatial dependence (diagnostic only)
Residuals of a **non-spatial RF** (all covariates, out-of-bag predictions) → empirical variogram → best of spherical / exponential / Gaussian (weighted LS). As in the R v4 workflow this is **not** used to size the blocks; it shows how much spatial dependence the covariates leave and is reported next to the block size.

In [ ]:
def vg_task(sc, y, d, cont, cat):
    warnings.simplefilter("ignore")
    feats = cont + cat; enc = RFEnc(feats, cat).fit(d, y); X = enc.transform(d); yy = d[y].values
    rf = RandomForestRegressor(n_estimators=300, max_features=max(1, len(feats) // 3), min_samples_leaf=3, oob_score=True, bootstrap=True, n_jobs=1, random_state=1).fit(X, yy)
    res = yy - rf.oob_prediction_; ok = np.isfinite(res)
    v = fit_variogram(d.loc[ok, ["px", "py"]].values, res[ok]); v.pop("fn", None)
    return (sc, y), dict(n=int(ok.sum()), **{k: v.get(k) for k in ["model", "nugget", "psill", "eff_range", "h", "gamma", "cutoff"]})
vg = dict(cached(f"variograms_{CFG_TAG}", lambda: Parallel(n_jobs=N_JOBS)(delayed(vg_task)(sc, y, scale_data(sc).dropna(subset=[y]), PRED_SETS[sc]["cont"], PRED_SETS[sc]["cat"]) for sc in SCALES for y in TARGETS)))
vg_tab = pd.DataFrame([dict(Scale=sc, Pool=TARGETS[y], n=o["n"], Model=o["model"], Nugget=o["nugget"], Sill=o["psill"],
                            Eff_range_m=(o["eff_range"] if (o["eff_range"] is not None and o["eff_range"] <= o["cutoff"]) else np.nan),   # ranges beyond the lag cutoff = no structure detected
                            Range_beyond_cutoff=bool(o["eff_range"] is not None and o["eff_range"] > o["cutoff"])) for (sc, y), o in vg.items()])
vg_tab = vg_tab.merge(block_diag[["Scale", "Design", "Autocorr_range_km"]].assign(Block_size_km=block_diag.Design.str.extract(r"([0-9.]+) km")[0].astype(float)), on="Scale", how="left")
vg_tab["Block_to_range"] = vg_tab.Block_size_km * 1000 / vg_tab.Eff_range_m
display(vg_tab.round(2))

def fig_variograms():
    fig, axes = plt.subplots(len(SCALES), len(TARGETS), figsize=(2.7 * len(TARGETS), 2.3 * len(SCALES)), squeeze=False)
    for i, sc in enumerate(SCALES):
        for j, y in enumerate(TARGETS):
            o = vg[(sc, y)]; ax = axes[i, j]
            if o["h"] is not None and len(o["h"]):
                ax.plot(o["h"] / 1000, o["gamma"], "o", ms=3, color=PAL_SCALE[sc])
                if o["eff_range"] is not None and np.isfinite(o["eff_range"]) and o["eff_range"] <= o["cutoff"]: ax.axvline(o["eff_range"] / 1000, ls="--", color="grey", lw=0.8)
            if i == 0: ax.set_title(TARGETS[y], fontsize=9)
            if j == 0: ax.set_ylabel(f"{sc}\nsemivariance")
            if i == len(SCALES) - 1: ax.set_xlabel("distance (km)")
            ax.tick_params(labelsize=7)
    fig.tight_layout(); save_fig(fig, "FigS_variograms"); plt.show()
fig_variograms()

# 11. Export

In [ ]:
sheets = {"CV_summary": cv_summary, "CV_overall_OOF": overall, "CV_fold_metrics": cv_fold, "Model_tests": model_tests, "Table7_MAE": tab7, "Best_model": best_tab,
          "Final_models": hp_final, "Importance": imp_tab, "Main_predictors": main_tab, "Totals_TgC": totals,
          "Block_design": block_diag, "Block_fold_sizes": fold_sizes, "Fold_plan": pd.concat([PLAN_ALL[s].assign(Site=s) for s in SITES]),
          "Block_search": pd.concat([v["summary"].assign(Site=s) for s, v in BLOCK_SEARCH.items()]) if BLOCK_SEARCH else pd.DataFrame({"Note": ["BLOCK_MODE = fixed"]}),
          "Variogram_diag": vg_tab, "Covariates": pd.DataFrame([dict(Scale=sc, Continuous=", ".join(v["cont"]), Categorical=", ".join(v["cat"])) for sc, v in PRED_SETS.items()]),
          "VIF": vif_wide, "Spearman_gt_0.8": rho_tab, "Raster_QA": qa, "Raster_gap_fill": fill_report, "Optuna_hp_per_fold": cv_hp}
with pd.ExcelWriter(TAB / "Results_nested_spatial_CV.xlsx") as xw:
    for k, v in sheets.items(): v.to_excel(xw, sheet_name=k[:31], index=False)
for k in ["CV_summary", "CV_overall_OOF", "Best_model", "Totals_TgC", "Block_design", "Final_models"]: sheets[k].to_csv(TAB / f"Table_{k}.csv", index=False)
cv_long.to_csv(TAB / "Table_CV_predictions.csv", index=False)
pd.concat([PLAN_ALL[s].assign(Site=s) for s in SITES]).to_csv(TAB / "folds_ALL_sites_combined_python.csv", index=False)
print("written to", OUT); print(sorted(p.name for p in (TAB).glob("*"))[:20])

# 12. Methods text (draft)
*Spatial blocking and evaluation.* Spatial blocks were designed for each site and the same design was used for all pools and algorithms. For every site, candidate designs (square and hexagonal blocks of several sizes, expressed as multiples of the spatial autocorrelation range of the pools and as fractions of the site extent, and k-means spatial clusters; random, balanced block-to-fold assignment over 100 iterations, five random assignments per design) were compared on how well the distance between test and training plots in the cross-validation reproduced the distance between map cells and the nearest sampled plot (Wasserstein distance between the two distributions, restricted to the sampled area plus a 10 % buffer; NNDM logic), on the share of test plots closer to a training plot than the autocorrelation range, and on the balance of the fold sizes. Among the designs within 0.03 of the best distance match, the one with the most balanced folds was retained. Designs were chosen on these geometric criteria only, never on model accuracy; the selected designs were square blocks of 11 km (AF), 3.5 km (AT) and 1.5 km (IV). Outer folds (K = 10) used the selected design; inner folds (K = 5) were rebuilt inside each outer training set with the same design, so that no information from an outer test fold entered variable selection or tuning. For the all-sites model, outer fold *k* was the union of the outer folds *k* of the three sites (and likewise for the inner folds).

*Models.* Random forest (scikit-learn; 500 trees; `mtry` and `min_samples_leaf` tuned with Optuna (TPE sampler, median pruner) on the inner spatial folds; variable selection by permutation importance on the held-out inner folds followed by a 1-SE rule), additive model with penalised cubic B-splines (second-order difference penalty; basis size and smoothing parameter tuned with Optuna; forward selection on the inner-CV RMSE) and linear mixed model (statsmodels; random intercept for site or spatial block; forward AIC selection; predictions from fixed effects only) were evaluated on identical folds with R², RMSE, nRMSE, MAE and RPD; mean, SD and t-based 95 % CI over the outer folds are reported and algorithms were compared with paired Wilcoxon signed-rank tests (Holm correction). The algorithm with the lowest rank sum of mean MAE and RMSE was used for mapping. *Uncertainty.* B = 100 bootstrap models (plots resampled with replacement within spatial blocks) gave per-pixel means, SDs and 2.5–97.5 % intervals and site totals (Tg C). Spatial dependence of the residuals of a non-spatial random forest (variograms) was used as a diagnostic.